In [1]:
import sys
import os
current_dir = os.path.abspath('')
os.chdir(current_dir)
sys.path.append(os.path.join(current_dir,'..','code','BalancingControl'))

import two_stage_utils as tu
import inference_utils as iu
import inference as inf

torch threads 1


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on device cpu
torch threads 1


In [2]:
import torch
import pyro

import pyro.distributions as dist

import os
from scipy.io import loadmat
import matplotlib.pylab as plt
import seaborn as sns
import pandas as pd
import glob
import pickle
import jsonpickle as jpickle
import json
import gc

In [3]:
results_folder = "results"
simulation_folder = os.path.join(results_folder, "simulations")
cross_fitting_folder = os.path.join(results_folder, "cross_fitting")
confusion_folder = os.path.join(results_folder, "confusion_matrix")
recovery_folder = os.path.join(results_folder, "recovery")

mask_file_name = "mask.txt"
processed_data_folder = os.path.join("processed_data")
mask_file = os.path.join(processed_data_folder, mask_file_name)

n_agents = 188

recalc_measures_BCC2_p = True
recalc_measures_BCC4_p_r = True
recalc_measures_BCC4_p_c = True
recalc_measures_BCC6_p_r_c = True
recalc_measures_MFMB4_mf_mb = True
recalc_measures_MFMB6_mf_mb_prior = True

WAIC_max_samples = 100

num_steps = 600

In [4]:
trials =  201#number of trials
T = 3 #number of time steps in each trial
nb = 4 # number of bandits, ie second level rewards
ns = 3+nb #number of states
no = ns #number of observations
na = 2 #number of actions
npi = na**(T-1) #number of policies
nr = 2 #number of rewards
never_reward = ns-nb # states that dont generate rewards

# prob for invalid answer (e.g. no reply). Same frequency as in the real data
with open(mask_file, "rb") as f:
    all_mask = pickle.load(f)
# simulations will be done with the same missing actions as in the data.
exp_mask = torch.tensor(all_mask).permute((1,0))
p_valid = exp_mask.sum()/(exp_mask.shape[0]*exp_mask.shape[1])
print(p_valid)

# make global parameter dict:
global_experiment_parameters = {"trials": trials, "T": T, "nb": nb, "ns": ns, "no": no, "na": na, "npi": npi, "nr": nr, "never_reward": never_reward, "p_invalid": p_valid, "mask": exp_mask}

tensor(0.9766)


/tmp/ipykernel_210181/1544998221.py:15: DeprecationWarning: In future, it will be an error for 'np.bool' scalars to be interpreted as an index
  exp_mask = torch.tensor(all_mask).permute((1,0))


In [5]:
#generating probability of observations in each state / unity matrix
A = torch.eye(no)


#state transition generative probability (matrix)
B = torch.zeros((ns, ns, na))
b1 = 0.7
nb1 = 1.-b1
b2 = 0.7
nb2 = 1.-b2

B[:,:,0] = torch.tensor([[  0,  0,  0,  0,  0,  0,  0,],
                         [ b1,  0,  0,  0,  0,  0,  0,],
                         [nb1,  0,  0,  0,  0,  0,  0,],
                         [  0,  1,  0,  1,  0,  0,  0,],
                         [  0,  0,  1,  0,  1,  0,  0,],
                         [  0,  0,  0,  0,  0,  1,  0,],
                         [  0,  0,  0,  0,  0,  0,  1,],])

B[:,:,1] = torch.tensor([[  0,  0,  0,  0,  0,  0,  0,],
                         [nb2,  0,  0,  0,  0,  0,  0,],
                         [ b2,  0,  0,  0,  0,  0,  0,],
                         [  0,  0,  0,  1,  0,  0,  0,],
                         [  0,  0,  0,  0,  1,  0,  0,],
                         [  0,  1,  0,  0,  0,  1,  0,],
                         [  0,  0,  1,  0,  0,  0,  1,],])

# add to parameter dict
global_experiment_parameters["A"] = A
global_experiment_parameters["B"] = B

In [6]:
def load_simulated_data(base_dir, agent_type):
    print("loading simulated outputs...")

    stayed_arr, true_vals, data = tu.load_simulation_outputs(base_dir, agent_type)

    n_true = true_vals["subject"].max() + 1
    n_data = data["subject"].max() + 1

    assert n_true == n_data == n_agents, f"the numbers of agents dont match! They are: {n_true}, {n_data}, {n_agents}. Probably rerun simulations to fix."

    print("true values are:")
    print(true_vals)

    return true_vals, data

In [7]:
def load_BCC_results(learn_rewards, learn_habit, learn_cached, use_h, base_dir, global_experiment_parameters, valid, fname_base, num_steps, data, param_names):    

    # set up agent
    bayes_agent = tu.set_up_Bayesian_inference_agent(n_agents, learn_rewards, learn_habit, learn_cached, base_dir, 
                                                    global_experiment_parameters, data["valid"], remove_old=False, 
                                                    use_h=use_h)

    print('analyzing '+str(n_agents)+' data sets')

    resample = False

    # set up inference
    inferrer = inf.GeneralGroupInference(bayes_agent, data)

    fname_str = fname_base + str(num_steps)+'_'+str(n_agents)+'agents'

    inferrer.load_parameters(os.path.join(base_dir, fname_str+"_parameter.save"))

    inferrer.load_elbo(os.path.join(base_dir, fname_str+"_elbo.save"))

    if resample:
        mean_df, sample_df, locs_df = iu.sample_posterior(inferrer, param_names, fname_str, base_dir) 
    else:
        mean_df, sample_df, locs_df = iu.load_samples(base_dir, fname_str) 

    return mean_df, sample_df, locs_df

In [8]:
def load_MFMB_results(learn_prior, use_orig, use_p, restrict_alpha, min_alpha, max_dt, base_dir, global_experiment_parameters, valid, fname_base, num_steps, data, param_names):    

    # set up agent
    mfmb_agent = tu.set_up_mbmf_inference_agent(n_agents, learn_prior, use_orig, use_p, restrict_alpha, max_dt, min_alpha, base_dir, global_experiment_parameters, valid, remove_old=False)

    print('analyzing '+str(n_agents)+' data sets')

    resample = False

    # set up inference
    inferrer = inf.GeneralGroupInference(mfmb_agent, data)

    fname_str = fname_base + str(num_steps)+'_'+str(n_agents)+'agents'

    inferrer.load_parameters(os.path.join(base_dir, fname_str+"_parameter.save"))

    inferrer.load_elbo(os.path.join(base_dir, fname_str+"_elbo.save"))

    if resample:
        mean_df, sample_df, locs_df = iu.sample_posterior(inferrer, param_names, fname_str, base_dir) 
    else:
        mean_df, sample_df, locs_df = iu.load_samples(base_dir, fname_str) 

    return mean_df, sample_df, locs_df

# Define agents and load simulated data

## BCC2 planning agent & data

In [9]:
# load BCC2 data

# set parameters and their names

learn_rewards = True
learn_habit = False
use_h = False
learn_cached = False

param_names = []
param_ranges = []

prefix = "BCC"
model_name = "Bayesian prior-based contextual control model"
n_pars = 0
agnt_str = ""

if learn_rewards:
    n_pars += 2
    param_names += ["dec temp", "reward rate"]
    param_ranges += [[0,8], [0,1]]
    agnt_str += "_planning"

if learn_habit:
    # infer_h = True
    # infer_policy_rate = True 
    n_pars += 2
    agnt_str += "_repetition"
    param_names += ["habitual tendency", "policy rate"]
    if use_h:
        agnt_str += "_h"
        param_ranges += [[0,1], [0,1]]
    else:
        agnt_str += "_weight"
        param_ranges += [[0,8], [0,1]]
# else:
#     infer_h = False
#     infer_policy_rate = False

if learn_cached:
    n_pars += 2
    param_names += ["cached weight", "cached rate"]
    param_ranges += [[0,8], [0,1]]
    agnt_str += "_cached"

assert n_pars > 0, "please turn any part of the agent on, it cannot run without any learning or inference."

# prepare for saving results
# make base filename and folder string
BCC2_p_agent_type = prefix+"_"+str(n_pars)+"pars"+agnt_str
print(BCC2_p_agent_type)
fname_base = BCC2_p_agent_type+"_simulation_"
print(fname_base)
# define folder where we want to save data
BCC2_p_data_base_dir = os.path.join(simulation_folder,fname_base[:-1])

BCC2_p_agent_params = {"learn_rewards": learn_rewards, "learn_habit": learn_habit, 
                       "learn_cached": learn_cached, "use_h": use_h,
                       "param_names": param_names, "param_ranges": param_ranges}

BCC2_p_true_vals, BCC2_p_data = load_simulated_data(BCC2_p_data_base_dir, BCC2_p_agent_type)

BCC_2pars_planning
BCC_2pars_planning_simulation_
loading simulated outputs...
true values are:
{'subject': tensor([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
          14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
          28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
          42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
          56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
          70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
          84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
          98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
         112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
         140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
         154, 155, 156,

## BCC4 planning repetition agent & data

In [10]:
# BCC4_planning_repetition agent

learn_rewards = True
learn_habit = True
use_h = False
learn_cached = False

param_names = []
param_ranges = []

prefix = "BCC"
model_name = "Bayesian prior-based contextual control model"
n_pars = 0
agnt_str = ""

if learn_rewards:
    n_pars += 2
    param_names += ["dec temp", "reward rate"]
    param_ranges += [[0,8], [0,1]]
    agnt_str += "_planning"

if learn_habit:
    # infer_h = True
    # infer_policy_rate = True 
    n_pars += 2
    agnt_str += "_repetition"
    param_names += ["habitual tendency", "policy rate"]
    if use_h:
        agnt_str += "_h"
        param_ranges += [[0,1], [0,1]]
    else:
        agnt_str += "_weight"
        param_ranges += [[0,8], [0,1]]
# else:
#     infer_h = False
#     infer_policy_rate = False

if learn_cached:
    n_pars += 2
    param_names += ["cached weight", "cached rate"]
    param_ranges += [[0,8], [0,1]]
    agnt_str += "_cached"

assert n_pars > 0, "please turn any part of the agent on, it cannot run without any learning or inference."

# prepare for saving results
# make base filename and folder string
BCC4_p_r_agent_type = prefix+"_"+str(n_pars)+"pars"+agnt_str
print(BCC4_p_r_agent_type)
fname_base = BCC4_p_r_agent_type+"_simulation_"
print(fname_base)
# define folder where we want to save data
BCC4_p_r_data_base_dir = os.path.join(simulation_folder,fname_base[:-1])

BCC4_p_r_agent_params = {"learn_rewards": learn_rewards, "learn_habit": learn_habit, 
                       "learn_cached": learn_cached, "use_h": use_h,
                       "param_names": param_names, "param_ranges": param_ranges}

BCC4_p_r_true_vals, BCC4_p_r_data = load_simulated_data(BCC4_p_r_data_base_dir, BCC4_p_r_agent_type)

BCC_4pars_planning_repetition_weight
BCC_4pars_planning_repetition_weight_simulation_
loading simulated outputs...
true values are:
{'subject': tensor([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
          14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
          28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
          42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
          56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
          70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
          84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
          98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
         112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
         140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 15

## BCC4 planning cached agent & data

In [11]:
# BCC4_planning_cached agent

learn_rewards = True
learn_habit = False
use_h = False
learn_cached = True

param_names = []
param_ranges = []

prefix = "BCC"
model_name = "Bayesian prior-based contextual control model"
n_pars = 0
agnt_str = ""

if learn_rewards:
    n_pars += 2
    param_names += ["dec temp", "reward rate"]
    param_ranges += [[0,8], [0,1]]
    agnt_str += "_planning"

if learn_habit:
    # infer_h = True
    # infer_policy_rate = True 
    n_pars += 2
    agnt_str += "_repetition"
    param_names += ["habitual tendency", "policy rate"]
    if use_h:
        agnt_str += "_h"
        param_ranges += [[0,1], [0,1]]
    else:
        agnt_str += "_weight"
        param_ranges += [[0,8], [0,1]]
# else:
#     infer_h = False
#     infer_policy_rate = False

if learn_cached:
    n_pars += 2
    param_names += ["cached weight", "cached rate"]
    param_ranges += [[0,8], [0,1]]
    agnt_str += "_cached"

assert n_pars > 0, "please turn any part of the agent on, it cannot run without any learning or inference."

# prepare for saving results
# make base filename and folder string
BCC4_p_c_agent_type = prefix+"_"+str(n_pars)+"pars"+agnt_str
print(BCC4_p_c_agent_type)
fname_base = BCC4_p_c_agent_type+"_simulation_"
print(fname_base)
# define folder where we want to save data
BCC4_p_c_data_base_dir = os.path.join(simulation_folder,fname_base[:-1])

BCC4_p_c_agent_params = {"learn_rewards": learn_rewards, "learn_habit": learn_habit, 
                       "learn_cached": learn_cached, "use_h": use_h,
                       "param_names": param_names, "param_ranges": param_ranges}

BCC4_p_c_true_vals, BCC4_p_c_data = load_simulated_data(BCC4_p_c_data_base_dir, BCC4_p_c_agent_type)

BCC_4pars_planning_cached
BCC_4pars_planning_cached_simulation_
loading simulated outputs...
true values are:
{'subject': tensor([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
          14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
          28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
          42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
          56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
          70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
          84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
          98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
         112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
         140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
         

## BCC6 planning repetition cached agent & data

In [12]:
# BCC4_planning_cached agent

learn_rewards = True
learn_habit = True
use_h = False
learn_cached = True

param_names = []
param_ranges = []

prefix = "BCC"
model_name = "Bayesian prior-based contextual control model"
n_pars = 0
agnt_str = ""

if learn_rewards:
    n_pars += 2
    param_names += ["dec temp", "reward rate"]
    param_ranges += [[0,8], [0,1]]
    agnt_str += "_planning"

if learn_habit:
    # infer_h = True
    # infer_policy_rate = True 
    n_pars += 2
    agnt_str += "_repetition"
    param_names += ["habitual tendency", "policy rate"]
    if use_h:
        agnt_str += "_h"
        param_ranges += [[0,1], [0,1]]
    else:
        agnt_str += "_weight"
        param_ranges += [[0,8], [0,1]]
# else:
#     infer_h = False
#     infer_policy_rate = False

if learn_cached:
    n_pars += 2
    param_names += ["cached weight", "cached rate"]
    param_ranges += [[0,8], [0,1]]
    agnt_str += "_cached"

assert n_pars > 0, "please turn any part of the agent on, it cannot run without any learning or inference."

# prepare for saving results
# make base filename and folder string
BCC6_p_r_c_agent_type = prefix+"_"+str(n_pars)+"pars"+agnt_str
print(BCC6_p_r_c_agent_type)
fname_base = BCC6_p_r_c_agent_type+"_simulation_"
print(fname_base)
# define folder where we want to save data
BCC6_p_r_c_data_base_dir = os.path.join(simulation_folder,fname_base[:-1])

BCC6_p_r_c_agent_params = {"learn_rewards": learn_rewards, "learn_habit": learn_habit, 
                       "learn_cached": learn_cached, "use_h": use_h,
                       "param_names": param_names, "param_ranges": param_ranges}

BCC6_p_r_c_true_vals, BCC6_p_r_c_data = load_simulated_data(BCC6_p_r_c_data_base_dir, BCC6_p_r_c_agent_type)

BCC_6pars_planning_repetition_weight_cached
BCC_6pars_planning_repetition_weight_cached_simulation_
loading simulated outputs...
true values are:
{'subject': tensor([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
          14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
          28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
          42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
          56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
          70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
          84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
          98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
         112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
         140, 141, 142, 143, 144, 145, 146, 147, 148

## MFMB4 MF MB agent & data

In [13]:
# MFMB4 analysis

# set parameters and their names

learn_prior = False

use_orig = False

use_p = False
restrict_alpha = False
max_dt = 6

n_pars = 4

prefix = "MFMB"
agnt_str = ""

if use_orig:
    agnt_str += "_mf_mb_Orig"
    param_names = ["discount", "learning rate", "dec temp", "weight"]
    model_name = "original original w and beta model"
    param_ranges = [[0,1], [0,1], [0,8], [0,1]]
else:
    agnt_str += "_mf_mb"
    param_names = ["mb weight", "discount", "mf weight", "learning rate"]
    model_name = "two beta mbmf model"
    param_ranges = [[0,max_dt], [0,1], [0,max_dt], [0,1]]


if learn_prior:
    n_pars += 2
    param_names += ["prior lr", "prior weight"]
    param_ranges += [[0,1], [0,max_dt]]
    agnt_str += "_prior"

if use_p:
    n_pars += 1
    agnt_str += "_p"
    param_names += ["repetition"]
    
if restrict_alpha:
    agnt_str += "_resticted"
    min_alpha = 0.1
else:
    min_alpha = 0

max_dt = 6

# prepare for saving results
# make base filename and folder string
MFMB4_mf_mb_agent_type = prefix+"_"+str(n_pars)+"pars"+agnt_str

MFMB4_mf_mb_agent_params = {"learn_prior": learn_prior, "use_p": use_p, "use_orig": use_orig, 
                            "restrict_alpha": restrict_alpha, "max_dt": max_dt, "min_alpha": min_alpha,
                            "param_names": param_names, "param_ranges": param_ranges}

MFMB4_mf_mb_data_fname_base = MFMB4_mf_mb_agent_type+"_simulation_"
print(MFMB4_mf_mb_data_fname_base)
# define folder where we want to save data
MFMB4_mf_mb_data_base_dir = os.path.join(simulation_folder,MFMB4_mf_mb_data_fname_base[:-1])

MFMB4_mf_mb_true_vals, MFMB4_mf_mb_data = load_simulated_data(MFMB4_mf_mb_data_base_dir, MFMB4_mf_mb_agent_type)

MFMB_4pars_mf_mb_simulation_
loading simulated outputs...
true values are:
{'subject': tensor([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
          14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
          28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
          42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
          56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
          70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
          84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
          98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
         112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
         140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
         154, 155, 156, 157, 158, 159, 160, 

## MFMB6 MF MB agent & data

In [14]:
# MFMB6 analysis

# set parameters and their names

learn_prior = True

use_orig = False

use_p = False
restrict_alpha = False
max_dt = 6

n_pars = 4

prefix = "MFMB"
agnt_str = ""

if use_orig:
    agnt_str += "_mf_mb_Orig"
    param_names = ["discount", "learning rate", "dec temp", "weight"]
    model_name = "original original w and beta model"
    param_ranges = [[0,1], [0,1], [0,8], [0,1]]
else:
    agnt_str += "_mf_mb"
    param_names = ["mb weight", "discount", "mf weight", "learning rate"]
    model_name = "two beta mbmf model"
    param_ranges = [[0,max_dt], [0,1], [0,max_dt], [0,1]]


if learn_prior:
    n_pars += 2
    param_names += ["prior lr", "prior weight"]
    param_ranges += [[0,1], [0,max_dt]]
    agnt_str += "_prior"

if use_p:
    n_pars += 1
    agnt_str += "_p"
    param_names += ["repetition"]
    
if restrict_alpha:
    agnt_str += "_resticted"
    min_alpha = 0.1
else:
    min_alpha = 0

# prepare for saving results
# make base filename and folder string
MFMB6_mf_mb_prior_agent_type = prefix+"_"+str(n_pars)+"pars"+agnt_str

MFMB6_mf_mb_prior_agent_params = {"learn_prior": learn_prior, "use_p": use_p, "use_orig": use_orig, 
                            "restrict_alpha": restrict_alpha, "max_dt": max_dt, "min_alpha": min_alpha,
                            "param_names": param_names, "param_ranges": param_ranges}

MFMB6_mf_mb_prior_data_fname_base = MFMB6_mf_mb_prior_agent_type+"_simulation_"
print(MFMB6_mf_mb_prior_data_fname_base)
# define folder where we want to save data
MFMB6_mf_mb_prior_data_base_dir = os.path.join(simulation_folder,MFMB6_mf_mb_prior_data_fname_base[:-1])

MFMB6_mf_mb_prior_true_vals, MFMB6_mf_mb_prior_data = load_simulated_data(MFMB6_mf_mb_prior_data_base_dir, MFMB6_mf_mb_prior_agent_type)

MFMB_6pars_mf_mb_prior_simulation_
loading simulated outputs...
true values are:
{'subject': tensor([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
          14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
          28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
          42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
          56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
          70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
          84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
          98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
         112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
         140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
         154, 155, 156, 157, 158, 159,

# 1. True agent = BCC2

## 1.1 BCC2 inference

In [15]:
# load BCC2 fitting of BCC2 data

BCC2_p_BCC2_p_fname_base = BCC2_p_agent_type+"_recovery_"
print(BCC2_p_BCC2_p_fname_base)
# define folder where we want to save data
BCC2_p_BCC2_p_base_dir = os.path.join(recovery_folder,BCC2_p_BCC2_p_fname_base[:-1])

BCC2_p_BCC2_p_mean_df, BCC2_p_BCC2_p_sample_df, BCC2_p_BCC2_p_locs_df = \
    load_BCC_results(BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                     BCC2_p_agent_params["learn_cached"], BCC2_p_agent_params["use_h"],
                     BCC2_p_BCC2_p_base_dir, global_experiment_parameters, exp_mask, BCC2_p_BCC2_p_fname_base, 
                     num_steps, BCC2_p_data, BCC2_p_agent_params["param_names"])

BCC2_p_BCC2_p_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                                                    BCC2_p_agent_params["learn_cached"], BCC2_p_BCC2_p_base_dir, global_experiment_parameters, 
                                                    BCC2_p_data["valid"], remove_old=False, use_h=BCC2_p_agent_params["use_h"])


BCC_2pars_planning_recovery_
2
analyzing 188 data sets


2


/home/sarah/src/TwoStageStrategies/Chen_et_al_data/../code/BalancingControl/perception.py:142: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3683.)
  self.big_trans_matrix = ar.stack([ar.stack([generative_model_states[:,:,policies[pi,t]] for pi in range(self.npi)]) for t in range(self.T-1)]).T.to(device)
/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/

In [16]:
# calculate or load model comparison measure

fname_BCC2_p_BCC2_p_WAIC = os.path.join(BCC2_p_BCC2_p_base_dir, BCC2_p_BCC2_p_fname_base+"_WAIC.json")

if recalc_measures_BCC2_p:
    BCC2_p_BCC2_p_WAIC = iu.calculate_waic(BCC2_p_data, BCC2_p_BCC2_p_agent, BCC2_p_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC2_p_BCC2_p_WAIC)
    with open(fname_BCC2_p_BCC2_p_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC2_p_BCC2_p_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC2_p_BCC2_p_WAIC = jpickle.decode(pickled_WAIC)


KeyboardInterrupt: 

In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_BCC2_p_log_like = os.path.join(BCC2_p_BCC2_p_base_dir, BCC2_p_BCC2_p_fname_base+"_log_likelihood.json")

if recalc_measures_BCC2_p:
    BCC2_p_BCC2_p_log_like = iu.calculate_log_likelihood(BCC2_p_data, BCC2_p_BCC2_p_agent, BCC2_p_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC2_p_BCC2_p_log_like)
    with open(fname_BCC2_p_BCC2_p_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC2_p_BCC2_p_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC2_p_BCC2_p_log_like = jpickle.decode(pickled_log_like)


## 1.2 BCC4 planning repetition inference

In [ ]:
BCC2_p_BCC4_p_r_fname_base = BCC2_p_agent_type+"_cross_fitting_"+BCC4_p_r_agent_type
print(BCC2_p_BCC4_p_r_fname_base)
# define folder where we want to save data
BCC2_p_BCC4_p_r_base_dir = os.path.join(cross_fitting_folder,BCC2_p_BCC4_p_r_fname_base[:-1])

BCC2_p_BCC4_p_r_mean_df, BCC2_p_BCC4_p_r_sample_df, BCC2_p_BCC4_p_r_locs_df = \
    load_BCC_results(BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                     BCC4_p_r_agent_params["learn_cached"], BCC4_p_r_agent_params["use_h"],
                     BCC2_p_BCC4_p_r_base_dir, global_experiment_parameters, exp_mask, BCC2_p_BCC4_p_r_fname_base, 
                     num_steps, BCC2_p_data, BCC4_p_r_agent_params["param_names"])

BCC2_p_BCC4_p_r_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                                                    BCC4_p_r_agent_params["learn_cached"], BCC2_p_BCC4_p_r_base_dir, global_experiment_parameters, 
                                                    BCC2_p_data["valid"], remove_old=False, use_h=BCC4_p_r_agent_params["use_h"])


BCC_2pars_planning_cross_fitting_BCC_4pars_planning_repetition_weight
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_BCC4_p_r_WAIC = os.path.join(BCC2_p_BCC4_p_r_base_dir, BCC2_p_BCC4_p_r_fname_base+"_WAIC.json")

if recalc_measures_BCC2_p:
    BCC2_p_BCC4_p_r_WAIC = iu.calculate_waic(BCC2_p_data, BCC2_p_BCC4_p_r_agent, BCC2_p_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC2_p_BCC4_p_r_WAIC)
    with open(fname_BCC2_p_BCC4_p_r_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC2_p_BCC4_p_r_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC2_p_BCC4_p_r_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_BCC4_p_r_log_like = os.path.join(BCC2_p_BCC4_p_r_base_dir, BCC2_p_BCC4_p_r_fname_base+"_log_likelihood.json")

if recalc_measures_BCC2_p:
    BCC2_p_BCC4_p_r_log_like = iu.calculate_log_likelihood(BCC2_p_data, BCC2_p_BCC4_p_r_agent, BCC2_p_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC2_p_BCC4_p_r_log_like)
    with open(fname_BCC2_p_BCC4_p_r_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC2_p_BCC4_p_r_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC2_p_BCC4_p_r_log_like = jpickle.decode(pickled_log_like)


## 1.3 BCC4 planning cached inference

In [ ]:
BCC2_p_BCC4_p_c_fname_base = BCC2_p_agent_type+"_cross_fitting_"+BCC4_p_c_agent_type
print(BCC2_p_BCC4_p_c_fname_base)
# define folder where we want to save data
BCC2_p_BCC4_p_c_base_dir = os.path.join(cross_fitting_folder,BCC2_p_BCC4_p_c_fname_base[:-1])

BCC2_p_BCC4_p_c_mean_df, BCC2_p_BCC4_p_c_sample_df, BCC2_p_BCC4_p_c_locs_df = \
    load_BCC_results(BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                     BCC4_p_c_agent_params["learn_cached"], BCC4_p_c_agent_params["use_h"],
                     BCC2_p_BCC4_p_c_base_dir, global_experiment_parameters, exp_mask, BCC2_p_BCC4_p_c_fname_base, 
                     num_steps, BCC2_p_data, BCC4_p_c_agent_params["param_names"])

BCC2_p_BCC4_p_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                                                    BCC4_p_c_agent_params["learn_cached"], BCC2_p_BCC4_p_c_base_dir, global_experiment_parameters, 
                                                    BCC2_p_data["valid"], remove_old=False, use_h=BCC4_p_c_agent_params["use_h"])


BCC_2pars_planning_cross_fitting_BCC_4pars_planning_cached
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_BCC4_p_c_WAIC = os.path.join(BCC2_p_BCC4_p_c_base_dir, BCC2_p_BCC4_p_c_fname_base+"_WAIC.json")

if recalc_measures_BCC2_p:
    BCC2_p_BCC4_p_c_WAIC = iu.calculate_waic(BCC2_p_data, BCC2_p_BCC4_p_c_agent, BCC2_p_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC2_p_BCC4_p_c_WAIC)
    with open(fname_BCC2_p_BCC4_p_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC2_p_BCC4_p_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC2_p_BCC4_p_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_BCC4_p_c_log_like = os.path.join(BCC2_p_BCC4_p_c_base_dir, BCC2_p_BCC4_p_c_fname_base+"_log_likelihood.json")

if recalc_measures_BCC2_p:
    BCC2_p_BCC4_p_c_log_like = iu.calculate_log_likelihood(BCC2_p_data, BCC2_p_BCC4_p_c_agent, BCC2_p_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC2_p_BCC4_p_c_log_like)
    with open(fname_BCC2_p_BCC4_p_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC2_p_BCC4_p_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC2_p_BCC4_p_c_log_like = jpickle.decode(pickled_log_like)


## 1.4 BCC6 planning repetition cached inference

In [ ]:
BCC2_p_BCC6_p_r_c_fname_base = BCC2_p_agent_type+"_cross_fitting_"+BCC6_p_r_c_agent_type
print(BCC2_p_BCC6_p_r_c_fname_base)
# define folder where we want to save data
BCC2_p_BCC6_p_r_c_base_dir = os.path.join(cross_fitting_folder,BCC2_p_BCC6_p_r_c_fname_base[:-1])

BCC2_p_BCC6_p_r_c_mean_df, BCC2_p_BCC6_p_r_c_sample_df, BCC2_p_BCC6_p_r_c_locs_df = \
    load_BCC_results(BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                     BCC6_p_r_c_agent_params["learn_cached"], BCC6_p_r_c_agent_params["use_h"],
                     BCC2_p_BCC6_p_r_c_base_dir, global_experiment_parameters, exp_mask, BCC2_p_BCC6_p_r_c_fname_base, 
                     num_steps, BCC2_p_data, BCC6_p_r_c_agent_params["param_names"])

BCC2_p_BCC6_p_r_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                                                    BCC6_p_r_c_agent_params["learn_cached"], BCC2_p_BCC6_p_r_c_base_dir, global_experiment_parameters, 
                                                    BCC2_p_data["valid"], remove_old=False, use_h=BCC6_p_r_c_agent_params["use_h"])


BCC_2pars_planning_cross_fitting_BCC_6pars_planning_repetition_weight_cached
6
analyzing 188 data sets
6


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_BCC6_p_r_c_WAIC = os.path.join(BCC2_p_BCC6_p_r_c_base_dir, BCC2_p_BCC6_p_r_c_fname_base+"_WAIC.json")

if recalc_measures_BCC2_p:
    BCC2_p_BCC6_p_r_c_WAIC = iu.calculate_waic(BCC2_p_data, BCC2_p_BCC6_p_r_c_agent, BCC2_p_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC2_p_BCC6_p_r_c_WAIC)
    with open(fname_BCC2_p_BCC6_p_r_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC2_p_BCC6_p_r_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC2_p_BCC6_p_r_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_BCC6_p_r_c_log_like = os.path.join(BCC2_p_BCC6_p_r_c_base_dir, BCC2_p_BCC6_p_r_c_fname_base+"_log_likelihood.json")

if recalc_measures_BCC2_p:
    BCC2_p_BCC6_p_r_c_log_like = iu.calculate_log_likelihood(BCC2_p_data, BCC2_p_BCC6_p_r_c_agent, BCC2_p_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC2_p_BCC6_p_r_c_log_like)
    with open(fname_BCC2_p_BCC6_p_r_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC2_p_BCC6_p_r_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC2_p_BCC6_p_r_c_log_like = jpickle.decode(pickled_log_like)


## 1.5 MFMB4 MF MB cached inference

In [ ]:
BCC2_p_MFMB4_mf_mb_fname_base = BCC2_p_agent_type+"_cross_fitting_"+MFMB4_mf_mb_agent_type
print(BCC2_p_MFMB4_mf_mb_fname_base)
# define folder where we want to save data
BCC2_p_MFMB4_mf_mb_base_dir = os.path.join(cross_fitting_folder,BCC2_p_MFMB4_mf_mb_fname_base[:-1])

BCC2_p_MFMB4_mf_mb_mean_df, BCC2_p_MFMB4_mf_mb_sample_df, BCC2_p_MFMB4_mf_mb_locs_df = \
    load_MFMB_results(MFMB4_mf_mb_agent_params["learn_prior"], MFMB4_mf_mb_agent_params["use_orig"], 
                      MFMB4_mf_mb_agent_params["use_p"], MFMB4_mf_mb_agent_params["restrict_alpha"], 
                      MFMB4_mf_mb_agent_params["min_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                      BCC2_p_MFMB4_mf_mb_base_dir, global_experiment_parameters, BCC2_p_data["valid"], 
                      BCC2_p_MFMB4_mf_mb_fname_base, num_steps, 
                      BCC2_p_data, MFMB4_mf_mb_agent_params["param_names"])

BCC2_p_MFMB4_mf_mb_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB4_mf_mb_agent_params["learn_prior"], 
                                                          MFMB4_mf_mb_agent_params["use_orig"], MFMB4_mf_mb_agent_params["use_p"], 
                                                          MFMB4_mf_mb_agent_params["restrict_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                                                          MFMB4_mf_mb_agent_params["min_alpha"], 
                                                          BCC2_p_MFMB4_mf_mb_base_dir, global_experiment_parameters, BCC2_p_data["valid"], remove_old=False)


BCC_2pars_planning_cross_fitting_MFMB_4pars_mf_mb
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_MFMB4_mf_mb_WAIC = os.path.join(BCC2_p_MFMB4_mf_mb_base_dir, BCC2_p_MFMB4_mf_mb_fname_base+"_WAIC.json")

if recalc_measures_BCC2_p:
    BCC2_p_MFMB4_mf_mb_WAIC = iu.calculate_waic(BCC2_p_data, BCC2_p_MFMB4_mf_mb_agent, BCC2_p_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC2_p_MFMB4_mf_mb_WAIC)
    with open(fname_BCC2_p_MFMB4_mf_mb_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC2_p_MFMB4_mf_mb_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC2_p_MFMB4_mf_mb_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_MFMB4_mf_mb_log_like = os.path.join(BCC2_p_MFMB4_mf_mb_base_dir, BCC2_p_MFMB4_mf_mb_fname_base+"_log_likelihood.json")

if recalc_measures_BCC2_p:
    BCC2_p_MFMB4_mf_mb_log_like = iu.calculate_log_likelihood(BCC2_p_data, BCC2_p_MFMB4_mf_mb_agent, BCC2_p_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC2_p_MFMB4_mf_mb_log_like)
    with open(fname_BCC2_p_MFMB4_mf_mb_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC2_p_MFMB4_mf_mb_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC2_p_MFMB4_mf_mb_log_like = jpickle.decode(pickled_log_like)


## 1.6 MFMB4 MF MB prior cached inference

In [ ]:
BCC2_p_MFMB6_mf_mb_prior_fname_base = BCC2_p_agent_type+"_cross_fitting_"+MFMB6_mf_mb_prior_agent_type
print(BCC2_p_MFMB6_mf_mb_prior_fname_base)
# define folder where we want to save data
BCC2_p_MFMB6_mf_mb_prior_base_dir = os.path.join(cross_fitting_folder,BCC2_p_MFMB6_mf_mb_prior_fname_base[:-1])

BCC2_p_MFMB6_mf_mb_prior_mean_df, BCC2_p_MFMB6_mf_mb_prior_sample_df, BCC2_p_MFMB6_mf_mb_prior_locs_df = \
    load_MFMB_results(MFMB6_mf_mb_prior_agent_params["learn_prior"], MFMB6_mf_mb_prior_agent_params["use_orig"], 
                      MFMB6_mf_mb_prior_agent_params["use_p"], MFMB6_mf_mb_prior_agent_params["restrict_alpha"], 
                      MFMB6_mf_mb_prior_agent_params["min_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                      BCC2_p_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, BCC2_p_data["valid"], 
                      BCC2_p_MFMB6_mf_mb_prior_fname_base, num_steps, 
                      BCC2_p_data, MFMB6_mf_mb_prior_agent_params["param_names"])

BCC2_p_MFMB6_mf_mb_prior_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB6_mf_mb_prior_agent_params["learn_prior"], 
                                                          MFMB6_mf_mb_prior_agent_params["use_orig"], MFMB6_mf_mb_prior_agent_params["use_p"], 
                                                          MFMB6_mf_mb_prior_agent_params["restrict_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                                                          MFMB6_mf_mb_prior_agent_params["min_alpha"], 
                                                          BCC2_p_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, BCC2_p_data["valid"], remove_old=False)


BCC_2pars_planning_cross_fitting_MFMB_6pars_mf_mb_prior
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_MFMB6_mf_mb_prior_WAIC = os.path.join(BCC2_p_MFMB6_mf_mb_prior_base_dir, BCC2_p_MFMB6_mf_mb_prior_fname_base+"_WAIC.json")

if recalc_measures_BCC2_p:
    BCC2_p_MFMB6_mf_mb_prior_WAIC = iu.calculate_waic(BCC2_p_data, BCC2_p_MFMB6_mf_mb_prior_agent, BCC2_p_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC2_p_MFMB6_mf_mb_prior_WAIC)
    with open(fname_BCC2_p_MFMB6_mf_mb_prior_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC2_p_MFMB6_mf_mb_prior_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC2_p_MFMB6_mf_mb_prior_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC2_p_MFMB6_mf_mb_prior_log_like = os.path.join(BCC2_p_MFMB6_mf_mb_prior_base_dir, BCC2_p_MFMB6_mf_mb_prior_fname_base+"_log_likelihood.json")

if recalc_measures_BCC2_p:
    BCC2_p_MFMB6_mf_mb_prior_log_like = iu.calculate_log_likelihood(BCC2_p_data, BCC2_p_MFMB6_mf_mb_prior_agent, BCC2_p_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC2_p_MFMB6_mf_mb_prior_log_like)
    with open(fname_BCC2_p_MFMB6_mf_mb_prior_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC2_p_MFMB6_mf_mb_prior_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC2_p_MFMB6_mf_mb_prior_log_like = jpickle.decode(pickled_log_like)


# 2. True agent = BCC4 planning repetition

## 2.1 BCC2 inference

In [ ]:
BCC4_p_r_BCC2_p_fname_base = BCC4_p_r_agent_type+"_cross_fitting_"+BCC2_p_agent_type
print(BCC4_p_r_BCC2_p_fname_base)
# define folder where we want to save data
BCC4_p_r_BCC2_p_base_dir = os.path.join(cross_fitting_folder,BCC4_p_r_BCC2_p_fname_base[:-1])

BCC4_p_r_BCC2_p_mean_df, BCC4_p_r_BCC2_p_sample_df, BCC4_p_r_BCC2_p_locs_df = \
    load_BCC_results(BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                     BCC2_p_agent_params["learn_cached"], BCC2_p_agent_params["use_h"],
                     BCC4_p_r_BCC2_p_base_dir, global_experiment_parameters, exp_mask, BCC4_p_r_BCC2_p_fname_base, 
                     num_steps, BCC4_p_r_data, BCC2_p_agent_params["param_names"])

BCC4_p_r_BCC2_p_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                                                    BCC2_p_agent_params["learn_cached"], BCC4_p_r_BCC2_p_base_dir, global_experiment_parameters, 
                                                    BCC4_p_r_data["valid"], remove_old=False, use_h=BCC2_p_agent_params["use_h"])


BCC_4pars_planning_repetition_weight_cross_fitting_BCC_2pars_planning
2
analyzing 188 data sets
2


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_BCC2_p_WAIC = os.path.join(BCC4_p_r_BCC2_p_base_dir, BCC4_p_r_BCC2_p_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_BCC2_p_WAIC = iu.calculate_waic(BCC4_p_r_data, BCC4_p_r_BCC2_p_agent, BCC4_p_r_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_r_BCC2_p_WAIC)
    with open(fname_BCC4_p_r_BCC2_p_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_r_BCC2_p_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_r_BCC2_p_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_BCC2_p_log_like = os.path.join(BCC4_p_r_BCC2_p_base_dir, BCC4_p_r_BCC2_p_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_BCC2_p_log_like = iu.calculate_log_likelihood(BCC4_p_r_data, BCC4_p_r_BCC2_p_agent, BCC4_p_r_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_r_BCC2_p_log_like)
    with open(fname_BCC4_p_r_BCC2_p_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_r_BCC2_p_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_r_BCC2_p_log_like = jpickle.decode(pickled_log_like)


## 2.2 BCC4 planning repetition inference

In [ ]:
# load BCC4 planning repetition fitting of BCC4 planning repetition data

BCC4_p_r_BCC4_p_r_fname_base = BCC4_p_r_agent_type+"_recovery_"
print(BCC4_p_r_BCC4_p_r_fname_base)
# define folder where we want to save data
BCC4_p_r_BCC4_p_r_base_dir = os.path.join(recovery_folder,BCC4_p_r_BCC4_p_r_fname_base[:-1])

BCC4_p_r_BCC4_p_r_mean_df, BCC4_p_r_BCC4_p_r_sample_df, BCC4_p_r_BCC4_p_r_locs_df = \
    load_BCC_results(BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                     BCC4_p_r_agent_params["learn_cached"], BCC4_p_r_agent_params["use_h"],
                     BCC4_p_r_BCC4_p_r_base_dir, global_experiment_parameters, exp_mask, BCC4_p_r_BCC4_p_r_fname_base, 
                     num_steps, BCC4_p_r_data, BCC4_p_r_agent_params["param_names"])

BCC4_p_r_BCC4_p_r_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                                                    BCC4_p_r_agent_params["learn_cached"], BCC4_p_r_BCC4_p_r_base_dir, global_experiment_parameters, 
                                                    BCC4_p_r_data["valid"], remove_old=False, use_h=BCC4_p_r_agent_params["use_h"])

BCC_4pars_planning_repetition_weight_recovery_
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_BCC4_p_r_WAIC = os.path.join(BCC4_p_r_BCC4_p_r_base_dir, BCC4_p_r_BCC4_p_r_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_BCC4_p_r_WAIC = iu.calculate_waic(BCC4_p_r_data, BCC4_p_r_BCC4_p_r_agent, BCC4_p_r_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_r_BCC4_p_r_WAIC)
    with open(fname_BCC4_p_r_BCC4_p_r_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_r_BCC4_p_r_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_r_BCC4_p_r_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_BCC4_p_r_log_like = os.path.join(BCC4_p_r_BCC4_p_r_base_dir, BCC4_p_r_BCC4_p_r_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_BCC4_p_r_log_like = iu.calculate_log_likelihood(BCC4_p_r_data, BCC4_p_r_BCC4_p_r_agent, BCC4_p_r_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_r_BCC4_p_r_log_like)
    with open(fname_BCC4_p_r_BCC4_p_r_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_r_BCC4_p_r_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_r_BCC4_p_r_log_like = jpickle.decode(pickled_log_like)


## 2.3 BCC4 planning cached inference

In [ ]:
BCC4_p_r_BCC4_p_c_fname_base = BCC4_p_r_agent_type+"_cross_fitting_"+BCC4_p_c_agent_type
print(BCC4_p_r_BCC4_p_c_fname_base)
# define folder where we want to save data
BCC4_p_r_BCC4_p_c_base_dir = os.path.join(cross_fitting_folder,BCC4_p_r_BCC4_p_c_fname_base[:-1])

BCC4_p_r_BCC4_p_c_mean_df, BCC4_p_r_BCC4_p_c_sample_df, BCC4_p_r_BCC4_p_c_locs_df = \
    load_BCC_results(BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                     BCC4_p_c_agent_params["learn_cached"], BCC4_p_c_agent_params["use_h"],
                     BCC4_p_r_BCC4_p_c_base_dir, global_experiment_parameters, exp_mask, BCC4_p_r_BCC4_p_c_fname_base, 
                     num_steps, BCC4_p_r_data, BCC4_p_c_agent_params["param_names"])

BCC4_p_r_BCC4_p_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                                                    BCC4_p_c_agent_params["learn_cached"], BCC4_p_r_BCC4_p_c_base_dir, global_experiment_parameters, 
                                                    BCC4_p_r_data["valid"], remove_old=False, use_h=BCC4_p_c_agent_params["use_h"])


BCC_4pars_planning_repetition_weight_cross_fitting_BCC_4pars_planning_cached
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_BCC4_p_c_WAIC = os.path.join(BCC4_p_r_BCC4_p_c_base_dir, BCC4_p_r_BCC4_p_c_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_BCC4_p_c_WAIC = iu.calculate_waic(BCC4_p_r_data, BCC4_p_r_BCC4_p_c_agent, BCC4_p_r_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_r_BCC4_p_c_WAIC)
    with open(fname_BCC4_p_r_BCC4_p_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_r_BCC4_p_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_r_BCC4_p_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_BCC4_p_c_log_like = os.path.join(BCC4_p_r_BCC4_p_c_base_dir, BCC4_p_r_BCC4_p_c_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_BCC4_p_c_log_like = iu.calculate_log_likelihood(BCC4_p_r_data, BCC4_p_r_BCC4_p_c_agent, BCC4_p_r_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_r_BCC4_p_c_log_like)
    with open(fname_BCC4_p_r_BCC4_p_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_r_BCC4_p_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_r_BCC4_p_c_log_like = jpickle.decode(pickled_log_like)


## 2.4 BCC6 planning repetition cached inference

In [ ]:
BCC4_p_r_BCC6_p_r_c_fname_base = BCC4_p_r_agent_type+"_cross_fitting_"+BCC6_p_r_c_agent_type
print(BCC4_p_r_BCC6_p_r_c_fname_base)
# define folder where we want to save data
BCC4_p_r_BCC6_p_r_c_base_dir = os.path.join(cross_fitting_folder,BCC4_p_r_BCC6_p_r_c_fname_base[:-1])

BCC4_p_r_BCC6_p_r_c_mean_df, BCC4_p_r_BCC6_p_r_c_sample_df, BCC4_p_r_BCC6_p_r_c_locs_df = \
    load_BCC_results(BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                     BCC6_p_r_c_agent_params["learn_cached"], BCC6_p_r_c_agent_params["use_h"],
                     BCC4_p_r_BCC6_p_r_c_base_dir, global_experiment_parameters, exp_mask, BCC4_p_r_BCC6_p_r_c_fname_base, 
                     num_steps, BCC4_p_r_data, BCC6_p_r_c_agent_params["param_names"])

BCC4_p_r_BCC6_p_r_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                                                    BCC6_p_r_c_agent_params["learn_cached"], BCC4_p_r_BCC6_p_r_c_base_dir, global_experiment_parameters, 
                                                    BCC4_p_r_data["valid"], remove_old=False, use_h=BCC6_p_r_c_agent_params["use_h"])


BCC_4pars_planning_repetition_weight_cross_fitting_BCC_6pars_planning_repetition_weight_cached
6
analyzing 188 data sets
6


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_BCC6_p_r_c_WAIC = os.path.join(BCC4_p_r_BCC6_p_r_c_base_dir, BCC4_p_r_BCC6_p_r_c_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_BCC6_p_r_c_WAIC = iu.calculate_waic(BCC4_p_r_data, BCC4_p_r_BCC6_p_r_c_agent, BCC4_p_r_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_r_BCC6_p_r_c_WAIC)
    with open(fname_BCC4_p_r_BCC6_p_r_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_r_BCC6_p_r_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_r_BCC6_p_r_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_BCC6_p_r_c_log_like = os.path.join(BCC4_p_r_BCC6_p_r_c_base_dir, BCC4_p_r_BCC6_p_r_c_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_BCC6_p_r_c_log_like = iu.calculate_log_likelihood(BCC4_p_r_data, BCC4_p_r_BCC6_p_r_c_agent, BCC4_p_r_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_r_BCC6_p_r_c_log_like)
    with open(fname_BCC4_p_r_BCC6_p_r_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_r_BCC6_p_r_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_r_BCC6_p_r_c_log_like = jpickle.decode(pickled_log_like)


## 2.5 MFMB4 MF MB cached inference

In [ ]:
BCC4_p_r_MFMB4_mf_mb_fname_base = BCC4_p_r_agent_type+"_cross_fitting_"+MFMB4_mf_mb_agent_type
print(BCC4_p_r_MFMB4_mf_mb_fname_base)
# define folder where we want to save data
BCC4_p_r_MFMB4_mf_mb_base_dir = os.path.join(cross_fitting_folder,BCC4_p_r_MFMB4_mf_mb_fname_base[:-1])

BCC4_p_r_MFMB4_mf_mb_mean_df, BCC4_p_r_MFMB4_mf_mb_sample_df, BCC4_p_r_MFMB4_mf_mb_locs_df = \
    load_MFMB_results(MFMB4_mf_mb_agent_params["learn_prior"], MFMB4_mf_mb_agent_params["use_orig"], 
                      MFMB4_mf_mb_agent_params["use_p"], MFMB4_mf_mb_agent_params["restrict_alpha"], 
                      MFMB4_mf_mb_agent_params["min_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                      BCC4_p_r_MFMB4_mf_mb_base_dir, global_experiment_parameters, BCC4_p_r_data["valid"], 
                      BCC4_p_r_MFMB4_mf_mb_fname_base, num_steps, 
                      BCC4_p_r_data, MFMB4_mf_mb_agent_params["param_names"])

BCC4_p_r_MFMB4_mf_mb_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB4_mf_mb_agent_params["learn_prior"], 
                                                          MFMB4_mf_mb_agent_params["use_orig"], MFMB4_mf_mb_agent_params["use_p"], 
                                                          MFMB4_mf_mb_agent_params["restrict_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                                                          MFMB4_mf_mb_agent_params["min_alpha"], 
                                                          BCC4_p_r_MFMB4_mf_mb_base_dir, global_experiment_parameters, BCC4_p_r_data["valid"], remove_old=False)


BCC_4pars_planning_repetition_weight_cross_fitting_MFMB_4pars_mf_mb
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_MFMB4_mf_mb_WAIC = os.path.join(BCC4_p_r_MFMB4_mf_mb_base_dir, BCC4_p_r_MFMB4_mf_mb_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_MFMB4_mf_mb_WAIC = iu.calculate_waic(BCC4_p_r_data, BCC4_p_r_MFMB4_mf_mb_agent, BCC4_p_r_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_r_MFMB4_mf_mb_WAIC)
    with open(fname_BCC4_p_r_MFMB4_mf_mb_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_r_MFMB4_mf_mb_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_r_MFMB4_mf_mb_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_MFMB4_mf_mb_log_like = os.path.join(BCC4_p_r_MFMB4_mf_mb_base_dir, BCC4_p_r_MFMB4_mf_mb_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_MFMB4_mf_mb_log_like = iu.calculate_log_likelihood(BCC4_p_r_data, BCC4_p_r_MFMB4_mf_mb_agent, BCC4_p_r_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_r_MFMB4_mf_mb_log_like)
    with open(fname_BCC4_p_r_MFMB4_mf_mb_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_r_MFMB4_mf_mb_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_r_MFMB4_mf_mb_log_like = jpickle.decode(pickled_log_like)


## 2.6 MFMB4 MF MB prior cached inference

In [ ]:
BCC4_p_r_MFMB6_mf_mb_prior_fname_base = BCC4_p_r_agent_type+"_cross_fitting_"+MFMB6_mf_mb_prior_agent_type
print(BCC4_p_r_MFMB6_mf_mb_prior_fname_base)
# define folder where we want to save data
BCC4_p_r_MFMB6_mf_mb_prior_base_dir = os.path.join(cross_fitting_folder,BCC4_p_r_MFMB6_mf_mb_prior_fname_base[:-1])

BCC4_p_r_MFMB6_mf_mb_prior_mean_df, BCC4_p_r_MFMB6_mf_mb_prior_sample_df, BCC4_p_r_MFMB6_mf_mb_prior_locs_df = \
    load_MFMB_results(MFMB6_mf_mb_prior_agent_params["learn_prior"], MFMB6_mf_mb_prior_agent_params["use_orig"], 
                      MFMB6_mf_mb_prior_agent_params["use_p"], MFMB6_mf_mb_prior_agent_params["restrict_alpha"], 
                      MFMB6_mf_mb_prior_agent_params["min_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                      BCC4_p_r_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, BCC4_p_r_data["valid"], 
                      BCC4_p_r_MFMB6_mf_mb_prior_fname_base, num_steps, 
                      BCC4_p_r_data, MFMB6_mf_mb_prior_agent_params["param_names"])

BCC4_p_r_MFMB6_mf_mb_prior_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB6_mf_mb_prior_agent_params["learn_prior"], 
                                                          MFMB6_mf_mb_prior_agent_params["use_orig"], MFMB6_mf_mb_prior_agent_params["use_p"], 
                                                          MFMB6_mf_mb_prior_agent_params["restrict_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                                                          MFMB6_mf_mb_prior_agent_params["min_alpha"], 
                                                          BCC4_p_r_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, BCC4_p_r_data["valid"], remove_old=False)


BCC_4pars_planning_repetition_weight_cross_fitting_MFMB_6pars_mf_mb_prior
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_MFMB6_mf_mb_prior_WAIC = os.path.join(BCC4_p_r_MFMB6_mf_mb_prior_base_dir, BCC4_p_r_MFMB6_mf_mb_prior_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_MFMB6_mf_mb_prior_WAIC = iu.calculate_waic(BCC4_p_r_data, BCC4_p_r_MFMB6_mf_mb_prior_agent, BCC4_p_r_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_r_MFMB6_mf_mb_prior_WAIC)
    with open(fname_BCC4_p_r_MFMB6_mf_mb_prior_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_r_MFMB6_mf_mb_prior_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_r_MFMB6_mf_mb_prior_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_r_MFMB6_mf_mb_prior_log_like = os.path.join(BCC4_p_r_MFMB6_mf_mb_prior_base_dir, BCC4_p_r_MFMB6_mf_mb_prior_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_r:
    BCC4_p_r_MFMB6_mf_mb_prior_log_like = iu.calculate_log_likelihood(BCC4_p_r_data, BCC4_p_r_MFMB6_mf_mb_prior_agent, BCC4_p_r_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_r_MFMB6_mf_mb_prior_log_like)
    with open(fname_BCC4_p_r_MFMB6_mf_mb_prior_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_r_MFMB6_mf_mb_prior_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_r_MFMB6_mf_mb_prior_log_like = jpickle.decode(pickled_log_like)


# 3. True agent = BCC4 planning cached

## 3.1 BCC2 inference

In [ ]:
BCC4_p_c_BCC2_p_fname_base = BCC4_p_c_agent_type+"_cross_fitting_"+BCC2_p_agent_type
print(BCC4_p_c_BCC2_p_fname_base)
# define folder where we want to save data
BCC4_p_c_BCC2_p_base_dir = os.path.join(cross_fitting_folder, BCC4_p_c_BCC2_p_fname_base[:-1])

BCC4_p_c_BCC2_p_mean_df, BCC4_p_c_BCC2_p_sample_df, BCC4_p_c_BCC2_p_locs_df = \
    load_BCC_results(BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                     BCC2_p_agent_params["learn_cached"], BCC2_p_agent_params["use_h"],
                     BCC4_p_c_BCC2_p_base_dir, global_experiment_parameters, exp_mask, BCC4_p_c_BCC2_p_fname_base, 
                     num_steps, BCC4_p_c_data, BCC2_p_agent_params["param_names"])

BCC4_p_c_BCC2_p_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                                                    BCC2_p_agent_params["learn_cached"], BCC4_p_c_BCC2_p_base_dir, global_experiment_parameters, 
                                                    BCC4_p_c_data["valid"], remove_old=False, use_h=BCC2_p_agent_params["use_h"])

BCC_4pars_planning_cached_cross_fitting_BCC_2pars_planning
2
analyzing 188 data sets
2


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_BCC2_p_WAIC = os.path.join(BCC4_p_c_BCC2_p_base_dir, BCC4_p_c_BCC2_p_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_BCC2_p_WAIC = iu.calculate_waic(BCC4_p_c_data, BCC4_p_c_BCC2_p_agent, BCC4_p_c_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_c_BCC2_p_WAIC)
    with open(fname_BCC4_p_c_BCC2_p_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_c_BCC2_p_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_c_BCC2_p_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_BCC2_p_log_like = os.path.join(BCC4_p_c_BCC2_p_base_dir, BCC4_p_c_BCC2_p_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_BCC2_p_log_like = iu.calculate_log_likelihood(BCC4_p_c_data, BCC4_p_c_BCC2_p_agent, BCC4_p_c_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_c_BCC2_p_log_like)
    with open(fname_BCC4_p_c_BCC2_p_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_c_BCC2_p_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_c_BCC2_p_log_like = jpickle.decode(pickled_log_like)


## 3.2 BCC4 planning repetition inference

In [ ]:
BCC4_p_c_BCC4_p_r_fname_base = BCC4_p_c_agent_type+"_cross_fitting_"+BCC4_p_r_agent_type
print(BCC4_p_c_BCC4_p_r_fname_base)
# define folder where we want to save data
BCC4_p_c_BCC4_p_r_base_dir = os.path.join(cross_fitting_folder,BCC4_p_c_BCC4_p_r_fname_base[:-1])

BCC4_p_c_BCC4_p_r_mean_df, BCC4_p_c_BCC4_p_r_sample_df, BCC4_p_c_BCC4_p_r_locs_df = \
    load_BCC_results(BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                     BCC4_p_r_agent_params["learn_cached"], BCC4_p_r_agent_params["use_h"],
                     BCC4_p_c_BCC4_p_r_base_dir, global_experiment_parameters, exp_mask, BCC4_p_c_BCC4_p_r_fname_base, 
                     num_steps, BCC4_p_c_data, BCC4_p_r_agent_params["param_names"])

BCC4_p_c_BCC4_p_r_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                                                    BCC4_p_r_agent_params["learn_cached"], BCC4_p_c_BCC4_p_r_base_dir, global_experiment_parameters, 
                                                    BCC4_p_c_data["valid"], remove_old=False, use_h=BCC4_p_r_agent_params["use_h"])


BCC_4pars_planning_cached_cross_fitting_BCC_4pars_planning_repetition_weight
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_BCC4_p_r_WAIC = os.path.join(BCC4_p_c_BCC4_p_r_base_dir, BCC4_p_c_BCC4_p_r_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_BCC4_p_r_WAIC = iu.calculate_waic(BCC4_p_c_data, BCC4_p_c_BCC4_p_r_agent, BCC4_p_c_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_c_BCC4_p_r_WAIC)
    with open(fname_BCC4_p_c_BCC4_p_r_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_c_BCC4_p_r_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_c_BCC4_p_r_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_BCC4_p_r_log_like = os.path.join(BCC4_p_c_BCC4_p_r_base_dir, BCC4_p_c_BCC4_p_r_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_BCC4_p_r_log_like = iu.calculate_log_likelihood(BCC4_p_c_data, BCC4_p_c_BCC4_p_r_agent, BCC4_p_c_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_c_BCC4_p_r_log_like)
    with open(fname_BCC4_p_c_BCC4_p_r_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_c_BCC4_p_r_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_c_BCC4_p_r_log_like = jpickle.decode(pickled_log_like)


## 3.3 BCC4 planning cached inference

In [ ]:
# load BCC2 fitting of BCC2 data

BCC4_p_c_BCC4_p_c_fname_base = BCC4_p_c_agent_type+"_recovery_"
print(BCC4_p_c_BCC4_p_c_fname_base)
# define folder where we want to save data
BCC4_p_c_BCC4_p_c_base_dir = os.path.join(recovery_folder,BCC4_p_c_BCC4_p_c_fname_base[:-1])

BCC4_p_c_BCC4_p_c_mean_df, BCC4_p_c_BCC4_p_c_sample_df, BCC4_p_c_BCC4_p_c_locs_df = \
    load_BCC_results(BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                     BCC4_p_c_agent_params["learn_cached"], BCC4_p_c_agent_params["use_h"],
                     BCC4_p_c_BCC4_p_c_base_dir, global_experiment_parameters, exp_mask, BCC4_p_c_BCC4_p_c_fname_base, 
                     num_steps, BCC4_p_c_data, BCC4_p_c_agent_params["param_names"])

BCC4_p_c_BCC4_p_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                                                    BCC4_p_c_agent_params["learn_cached"], BCC4_p_c_BCC4_p_c_base_dir, global_experiment_parameters, 
                                                    BCC4_p_c_data["valid"], remove_old=False, use_h=BCC4_p_c_agent_params["use_h"])


BCC_4pars_planning_cached_recovery_
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_BCC4_p_c_WAIC = os.path.join(BCC4_p_c_BCC4_p_c_base_dir, BCC4_p_c_BCC4_p_c_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_BCC4_p_c_WAIC = iu.calculate_waic(BCC4_p_c_data, BCC4_p_c_BCC4_p_c_agent, BCC4_p_c_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_c_BCC4_p_c_WAIC)
    with open(fname_BCC4_p_c_BCC4_p_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_c_BCC4_p_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_c_BCC4_p_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_BCC4_p_c_log_like = os.path.join(BCC4_p_c_BCC4_p_c_base_dir, BCC4_p_c_BCC4_p_c_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_BCC4_p_c_log_like = iu.calculate_log_likelihood(BCC4_p_c_data, BCC4_p_c_BCC4_p_c_agent, BCC4_p_c_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like= jpickle.encode(BCC4_p_c_BCC4_p_c_log_like)
    with open(fname_BCC4_p_c_BCC4_p_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_c_BCC4_p_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_c_BCC4_p_c_log_like = jpickle.decode(pickled_log_like)


## 3.4 BCC6 planning repetition cached inference

In [ ]:
BCC4_p_c_BCC6_p_r_c_fname_base = BCC4_p_c_agent_type+"_cross_fitting_"+BCC6_p_r_c_agent_type
print(BCC4_p_c_BCC6_p_r_c_fname_base)
# define folder where we want to save data
BCC4_p_c_BCC6_p_r_c_base_dir = os.path.join(cross_fitting_folder,BCC4_p_c_BCC6_p_r_c_fname_base[:-1])

BCC4_p_c_BCC6_p_r_c_mean_df, BCC4_p_c_BCC6_p_r_c_sample_df, BCC4_p_c_BCC6_p_r_c_locs_df = \
    load_BCC_results(BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                     BCC6_p_r_c_agent_params["learn_cached"], BCC6_p_r_c_agent_params["use_h"],
                     BCC4_p_c_BCC6_p_r_c_base_dir, global_experiment_parameters, exp_mask, BCC4_p_c_BCC6_p_r_c_fname_base, 
                     num_steps, BCC4_p_c_data, BCC6_p_r_c_agent_params["param_names"])

BCC4_p_c_BCC6_p_r_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                                                    BCC6_p_r_c_agent_params["learn_cached"], BCC4_p_c_BCC6_p_r_c_base_dir, global_experiment_parameters, 
                                                    BCC4_p_c_data["valid"], remove_old=False, use_h=BCC6_p_r_c_agent_params["use_h"])


BCC_4pars_planning_cached_cross_fitting_BCC_6pars_planning_repetition_weight_cached
6
analyzing 188 data sets
6


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_BCC6_p_r_c_WAIC = os.path.join(BCC4_p_c_BCC6_p_r_c_base_dir, BCC4_p_c_BCC6_p_r_c_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_BCC6_p_r_c_WAIC = iu.calculate_waic(BCC4_p_c_data, BCC4_p_c_BCC6_p_r_c_agent, BCC4_p_c_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_c_BCC6_p_r_c_WAIC)
    with open(fname_BCC4_p_c_BCC6_p_r_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_c_BCC6_p_r_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_c_BCC6_p_r_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_BCC6_p_r_c_log_like = os.path.join(BCC4_p_c_BCC6_p_r_c_base_dir, BCC4_p_c_BCC6_p_r_c_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_BCC6_p_r_c_log_like = iu.calculate_log_likelihood(BCC4_p_c_data, BCC4_p_c_BCC6_p_r_c_agent, BCC4_p_c_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_c_BCC6_p_r_c_log_like)
    with open(fname_BCC4_p_c_BCC6_p_r_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_c_BCC6_p_r_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_c_BCC6_p_r_c_log_like = jpickle.decode(pickled_log_like)


## 3.5 MFMB4 MF MB cached inference

In [ ]:
BCC4_p_c_MFMB4_mf_mb_fname_base = BCC4_p_c_agent_type+"_cross_fitting_"+MFMB4_mf_mb_agent_type
print(BCC4_p_c_MFMB4_mf_mb_fname_base)
# define folder where we want to save data
BCC4_p_c_MFMB4_mf_mb_base_dir = os.path.join(cross_fitting_folder,BCC4_p_c_MFMB4_mf_mb_fname_base[:-1])

BCC4_p_c_MFMB4_mf_mb_mean_df, BCC4_p_c_MFMB4_mf_mb_sample_df, BCC4_p_c_MFMB4_mf_mb_locs_df = \
    load_MFMB_results(MFMB4_mf_mb_agent_params["learn_prior"], MFMB4_mf_mb_agent_params["use_orig"], 
                      MFMB4_mf_mb_agent_params["use_p"], MFMB4_mf_mb_agent_params["restrict_alpha"], 
                      MFMB4_mf_mb_agent_params["min_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                      BCC4_p_c_MFMB4_mf_mb_base_dir, global_experiment_parameters, BCC4_p_c_data["valid"], 
                      BCC4_p_c_MFMB4_mf_mb_fname_base, num_steps, 
                      BCC4_p_c_data, MFMB4_mf_mb_agent_params["param_names"])

BCC4_p_c_MFMB4_mf_mb_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB4_mf_mb_agent_params["learn_prior"], 
                                                          MFMB4_mf_mb_agent_params["use_orig"], MFMB4_mf_mb_agent_params["use_p"], 
                                                          MFMB4_mf_mb_agent_params["restrict_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                                                          MFMB4_mf_mb_agent_params["min_alpha"], 
                                                          BCC4_p_c_MFMB4_mf_mb_base_dir, global_experiment_parameters, BCC4_p_c_data["valid"], remove_old=False)


BCC_4pars_planning_cached_cross_fitting_MFMB_4pars_mf_mb
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_MFMB4_mf_mb_WAIC = os.path.join(BCC4_p_c_MFMB4_mf_mb_base_dir, BCC4_p_c_MFMB4_mf_mb_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_MFMB4_mf_mb_WAIC = iu.calculate_waic(BCC4_p_c_data, BCC4_p_c_MFMB4_mf_mb_agent, BCC4_p_c_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_c_MFMB4_mf_mb_WAIC)
    with open(fname_BCC4_p_c_MFMB4_mf_mb_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_c_MFMB4_mf_mb_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_c_MFMB4_mf_mb_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_MFMB4_mf_mb_log_like = os.path.join(BCC4_p_c_MFMB4_mf_mb_base_dir, BCC4_p_c_MFMB4_mf_mb_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_MFMB4_mf_mb_log_like = iu.calculate_log_likelihood(BCC4_p_c_data, BCC4_p_c_MFMB4_mf_mb_agent, BCC4_p_c_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_c_MFMB4_mf_mb_log_like)
    with open(fname_BCC4_p_c_MFMB4_mf_mb_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_c_MFMB4_mf_mb_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_c_MFMB4_mf_mb_log_like = jpickle.decode(pickled_log_like)


## 3.6 MFMB4 MF MB prior cached inference

In [ ]:
BCC4_p_c_MFMB6_mf_mb_prior_fname_base = BCC4_p_c_agent_type+"_cross_fitting_"+MFMB6_mf_mb_prior_agent_type
print(BCC4_p_c_MFMB6_mf_mb_prior_fname_base)
# define folder where we want to save data
BCC4_p_c_MFMB6_mf_mb_prior_base_dir = os.path.join(cross_fitting_folder,BCC4_p_c_MFMB6_mf_mb_prior_fname_base[:-1])

BCC4_p_c_MFMB6_mf_mb_prior_mean_df, BCC4_p_c_MFMB6_mf_mb_prior_sample_df, BCC4_p_c_MFMB6_mf_mb_prior_locs_df = \
    load_MFMB_results(MFMB6_mf_mb_prior_agent_params["learn_prior"], MFMB6_mf_mb_prior_agent_params["use_orig"], 
                      MFMB6_mf_mb_prior_agent_params["use_p"], MFMB6_mf_mb_prior_agent_params["restrict_alpha"], 
                      MFMB6_mf_mb_prior_agent_params["min_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                      BCC4_p_c_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, BCC4_p_c_data["valid"], 
                      BCC4_p_c_MFMB6_mf_mb_prior_fname_base, num_steps, 
                      BCC4_p_c_data, MFMB6_mf_mb_prior_agent_params["param_names"])

BCC4_p_c_MFMB6_mf_mb_prior_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB6_mf_mb_prior_agent_params["learn_prior"], 
                                                          MFMB6_mf_mb_prior_agent_params["use_orig"], MFMB6_mf_mb_prior_agent_params["use_p"], 
                                                          MFMB6_mf_mb_prior_agent_params["restrict_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                                                          MFMB6_mf_mb_prior_agent_params["min_alpha"], 
                                                          BCC4_p_c_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, BCC4_p_c_data["valid"], remove_old=False)


BCC_4pars_planning_cached_cross_fitting_MFMB_6pars_mf_mb_prior
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_MFMB6_mf_mb_prior_WAIC = os.path.join(BCC4_p_c_MFMB6_mf_mb_prior_base_dir, BCC4_p_c_MFMB6_mf_mb_prior_fname_base+"_WAIC.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_MFMB6_mf_mb_prior_WAIC = iu.calculate_waic(BCC4_p_c_data, BCC4_p_c_MFMB6_mf_mb_prior_agent, BCC4_p_c_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC4_p_c_MFMB6_mf_mb_prior_WAIC)
    with open(fname_BCC4_p_c_MFMB6_mf_mb_prior_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC4_p_c_MFMB6_mf_mb_prior_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC4_p_c_MFMB6_mf_mb_prior_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC4_p_c_MFMB6_mf_mb_prior_log_like = os.path.join(BCC4_p_c_MFMB6_mf_mb_prior_base_dir, BCC4_p_c_MFMB6_mf_mb_prior_fname_base+"_log_likelihood.json")

if recalc_measures_BCC4_p_c:
    BCC4_p_c_MFMB6_mf_mb_prior_log_like = iu.calculate_log_likelihood(BCC4_p_c_data, BCC4_p_c_MFMB6_mf_mb_prior_agent, BCC4_p_c_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC4_p_c_MFMB6_mf_mb_prior_log_like)
    with open(fname_BCC4_p_c_MFMB6_mf_mb_prior_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC4_p_c_MFMB6_mf_mb_prior_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC4_p_c_MFMB6_mf_mb_prior_log_like = jpickle.decode(pickled_log_like)


# 4. True agent = BCC6 planning repetition cached

## 4.1 BCC2 inference

In [ ]:
BCC6_p_r_c_BCC2_p_fname_base = BCC6_p_r_c_agent_type+"_cross_fitting_"+BCC2_p_agent_type
print(BCC6_p_r_c_BCC2_p_fname_base)
# define folder where we want to save data
BCC6_p_r_c_BCC2_p_base_dir = os.path.join(cross_fitting_folder,BCC6_p_r_c_BCC2_p_fname_base[:-1])

BCC6_p_r_c_BCC2_p_mean_df, BCC6_p_r_c_BCC2_p_sample_df, BCC6_p_r_c_BCC2_p_locs_df = \
    load_BCC_results(BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                     BCC2_p_agent_params["learn_cached"], BCC2_p_agent_params["use_h"],
                     BCC6_p_r_c_BCC2_p_base_dir, global_experiment_parameters, exp_mask, BCC6_p_r_c_BCC2_p_fname_base, 
                     num_steps, BCC6_p_r_c_data, BCC2_p_agent_params["param_names"])

BCC6_p_r_c_BCC2_p_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                                                    BCC2_p_agent_params["learn_cached"], BCC6_p_r_c_BCC2_p_base_dir, global_experiment_parameters, 
                                                    BCC6_p_r_c_data["valid"], remove_old=False, use_h=BCC2_p_agent_params["use_h"])

BCC_6pars_planning_repetition_weight_cached_cross_fitting_BCC_2pars_planning
2
analyzing 188 data sets
2


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_BCC2_p_WAIC = os.path.join(BCC6_p_r_c_BCC2_p_base_dir, BCC6_p_r_c_BCC2_p_fname_base+"_WAIC.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_BCC2_p_WAIC = iu.calculate_waic(BCC6_p_r_c_data, BCC6_p_r_c_BCC2_p_agent, BCC6_p_r_c_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC6_p_r_c_BCC2_p_WAIC)
    with open(fname_BCC6_p_r_c_BCC2_p_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC6_p_r_c_BCC2_p_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC6_p_r_c_BCC2_p_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_BCC2_p_log_like = os.path.join(BCC6_p_r_c_BCC2_p_base_dir, BCC6_p_r_c_BCC2_p_fname_base+"_log_like.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_BCC2_p_log_like = iu.calculate_log_likelihood(BCC6_p_r_c_data, BCC6_p_r_c_BCC2_p_agent, BCC6_p_r_c_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC6_p_r_c_BCC2_p_log_like)
    with open(fname_BCC6_p_r_c_BCC2_p_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC6_p_r_c_BCC2_p_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC6_p_r_c_BCC2_p_log_like = jpickle.decode(pickled_log_like)


## 4.2 BCC4 planning repetition inference

In [ ]:
BCC6_p_r_c_BCC4_p_r_fname_base = BCC6_p_r_c_agent_type+"_cross_fitting_"+BCC4_p_r_agent_type
print(BCC6_p_r_c_BCC4_p_r_fname_base)
# define folder where we want to save data
BCC6_p_r_c_BCC4_p_r_base_dir = os.path.join(cross_fitting_folder,BCC6_p_r_c_BCC4_p_r_fname_base[:-1])

BCC6_p_r_c_BCC4_p_r_mean_df, BCC6_p_r_c_BCC4_p_r_sample_df, BCC6_p_r_c_BCC4_p_r_locs_df = \
    load_BCC_results(BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                     BCC4_p_r_agent_params["learn_cached"], BCC4_p_r_agent_params["use_h"],
                     BCC6_p_r_c_BCC4_p_r_base_dir, global_experiment_parameters, exp_mask, BCC6_p_r_c_BCC4_p_r_fname_base, 
                     num_steps, BCC6_p_r_c_data, BCC4_p_r_agent_params["param_names"])

BCC6_p_r_c_BCC4_p_r_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                                                    BCC4_p_r_agent_params["learn_cached"], BCC6_p_r_c_BCC4_p_r_base_dir, global_experiment_parameters, 
                                                    BCC6_p_r_c_data["valid"], remove_old=False, use_h=BCC4_p_r_agent_params["use_h"])


BCC_6pars_planning_repetition_weight_cached_cross_fitting_BCC_4pars_planning_repetition_weight
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_BCC4_p_r_WAIC = os.path.join(BCC6_p_r_c_BCC4_p_r_base_dir, BCC6_p_r_c_BCC4_p_r_fname_base+"_WAIC.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_BCC4_p_r_WAIC = iu.calculate_waic(BCC6_p_r_c_data, BCC6_p_r_c_BCC4_p_r_agent, BCC6_p_r_c_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC6_p_r_c_BCC4_p_r_WAIC)
    with open(fname_BCC6_p_r_c_BCC4_p_r_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC6_p_r_c_BCC4_p_r_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC6_p_r_c_BCC4_p_r_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_BCC4_p_r_log_like = os.path.join(BCC6_p_r_c_BCC4_p_r_base_dir, BCC6_p_r_c_BCC4_p_r_fname_base+"_log_likelihood.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_BCC4_p_r_log_like = iu.calculate_log_likelihood(BCC6_p_r_c_data, BCC6_p_r_c_BCC4_p_r_agent, BCC6_p_r_c_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC6_p_r_c_BCC4_p_r_log_like)
    with open(fname_BCC6_p_r_c_BCC4_p_r_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC6_p_r_c_BCC4_p_r_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC6_p_r_c_BCC4_p_r_log_like = jpickle.decode(pickled_log_like)


## 4.3 BCC4 planning cached inference

In [ ]:
BCC6_p_r_c_BCC4_p_c_fname_base = BCC6_p_r_c_agent_type+"_cross_fitting_"+BCC4_p_c_agent_type
print(BCC6_p_r_c_BCC4_p_c_fname_base)
# define folder where we want to save data
BCC6_p_r_c_BCC4_p_c_base_dir = os.path.join(cross_fitting_folder,BCC6_p_r_c_BCC4_p_c_fname_base[:-1])

BCC6_p_r_c_BCC4_p_c_mean_df, BCC6_p_r_c_BCC4_p_c_sample_df, BCC6_p_r_c_BCC4_p_c_locs_df = \
    load_BCC_results(BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                     BCC4_p_c_agent_params["learn_cached"], BCC4_p_c_agent_params["use_h"],
                     BCC6_p_r_c_BCC4_p_c_base_dir, global_experiment_parameters, exp_mask, BCC6_p_r_c_BCC4_p_c_fname_base, 
                     num_steps, BCC6_p_r_c_data, BCC4_p_c_agent_params["param_names"])

BCC6_p_r_c_BCC4_p_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                                                    BCC4_p_c_agent_params["learn_cached"], BCC6_p_r_c_BCC4_p_c_base_dir, global_experiment_parameters, 
                                                    BCC6_p_r_c_data["valid"], remove_old=False, use_h=BCC4_p_c_agent_params["use_h"])


BCC_6pars_planning_repetition_weight_cached_cross_fitting_BCC_4pars_planning_cached
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_BCC4_p_c_WAIC = os.path.join(BCC6_p_r_c_BCC4_p_c_base_dir, BCC6_p_r_c_BCC4_p_c_fname_base+"_WAIC.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_BCC4_p_c_WAIC = iu.calculate_waic(BCC6_p_r_c_data, BCC6_p_r_c_BCC4_p_c_agent, BCC6_p_r_c_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC6_p_r_c_BCC4_p_c_WAIC)
    with open(fname_BCC6_p_r_c_BCC4_p_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC6_p_r_c_BCC4_p_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC6_p_r_c_BCC4_p_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_BCC4_p_c_log_like = os.path.join(BCC6_p_r_c_BCC4_p_c_base_dir, BCC6_p_r_c_BCC4_p_c_fname_base+"_log_likelihood.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_BCC4_p_c_log_like = iu.calculate_log_likelihood(BCC6_p_r_c_data, BCC6_p_r_c_BCC4_p_c_agent, BCC6_p_r_c_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC6_p_r_c_BCC4_p_c_log_like)
    with open(fname_BCC6_p_r_c_BCC4_p_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC6_p_r_c_BCC4_p_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC6_p_r_c_BCC4_p_c_log_like = jpickle.decode(pickled_log_like)


## 4.4 BCC6 planning repetition cached inference

In [ ]:
# load BCC2 fitting of BCC2 data

BCC6_p_r_c_BCC6_p_r_c_fname_base = BCC6_p_r_c_agent_type+"_recovery_"
print(BCC6_p_r_c_BCC6_p_r_c_fname_base)
# define folder where we want to save data
BCC6_p_r_c_BCC6_p_r_c_base_dir = os.path.join(recovery_folder,BCC6_p_r_c_BCC6_p_r_c_fname_base[:-1])

BCC6_p_r_c_BCC6_p_r_c_mean_df, BCC6_p_r_c_BCC6_p_r_c_sample_df, BCC6_p_r_c_BCC6_p_r_c_locs_df = \
    load_BCC_results(BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                     BCC6_p_r_c_agent_params["learn_cached"], BCC6_p_r_c_agent_params["use_h"],
                     BCC6_p_r_c_BCC6_p_r_c_base_dir, global_experiment_parameters, exp_mask, BCC6_p_r_c_BCC6_p_r_c_fname_base, 
                     num_steps, BCC6_p_r_c_data, BCC6_p_r_c_agent_params["param_names"])

BCC6_p_r_c_BCC6_p_r_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                                                    BCC6_p_r_c_agent_params["learn_cached"], BCC6_p_r_c_BCC6_p_r_c_base_dir, global_experiment_parameters, 
                                                    BCC6_p_r_c_data["valid"], remove_old=False, use_h=BCC6_p_r_c_agent_params["use_h"])


BCC_6pars_planning_repetition_weight_cached_recovery_
6
analyzing 188 data sets
6


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_BCC6_p_r_c_WAIC = os.path.join(BCC6_p_r_c_BCC6_p_r_c_base_dir, BCC6_p_r_c_BCC6_p_r_c_fname_base+"_WAIC.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_BCC6_p_r_c_WAIC = iu.calculate_waic(BCC6_p_r_c_data, BCC6_p_r_c_BCC6_p_r_c_agent, BCC6_p_r_c_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC6_p_r_c_BCC6_p_r_c_WAIC)
    with open(fname_BCC6_p_r_c_BCC6_p_r_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC6_p_r_c_BCC6_p_r_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC6_p_r_c_BCC6_p_r_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_BCC6_p_r_c_log_like = os.path.join(BCC6_p_r_c_BCC6_p_r_c_base_dir, BCC6_p_r_c_BCC6_p_r_c_fname_base+"_log_likelihood.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_BCC6_p_r_c_log_like = iu.calculate_log_likelihood(BCC6_p_r_c_data, BCC6_p_r_c_BCC6_p_r_c_agent, BCC6_p_r_c_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC6_p_r_c_BCC6_p_r_c_log_like)
    with open(fname_BCC6_p_r_c_BCC6_p_r_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC6_p_r_c_BCC6_p_r_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC6_p_r_c_BCC6_p_r_c_log_like = jpickle.decode(pickled_log_like)


## 4.5 MFMB4 MF MB cached inference

In [ ]:
BCC6_p_r_c_MFMB4_mf_mb_fname_base = BCC6_p_r_c_agent_type+"_cross_fitting_"+MFMB4_mf_mb_agent_type
print(BCC6_p_r_c_MFMB4_mf_mb_fname_base)
# define folder where we want to save data
BCC6_p_r_c_MFMB4_mf_mb_base_dir = os.path.join(cross_fitting_folder,BCC6_p_r_c_MFMB4_mf_mb_fname_base[:-1])

BCC6_p_r_c_MFMB4_mf_mb_mean_df, BCC6_p_r_c_MFMB4_mf_mb_sample_df, BCC6_p_r_c_MFMB4_mf_mb_locs_df = \
    load_MFMB_results(MFMB4_mf_mb_agent_params["learn_prior"], MFMB4_mf_mb_agent_params["use_orig"], 
                      MFMB4_mf_mb_agent_params["use_p"], MFMB4_mf_mb_agent_params["restrict_alpha"], 
                      MFMB4_mf_mb_agent_params["min_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                      BCC6_p_r_c_MFMB4_mf_mb_base_dir, global_experiment_parameters, BCC6_p_r_c_data["valid"], 
                      BCC6_p_r_c_MFMB4_mf_mb_fname_base, num_steps, 
                      BCC6_p_r_c_data, MFMB4_mf_mb_agent_params["param_names"])

BCC6_p_r_c_MFMB4_mf_mb_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB4_mf_mb_agent_params["learn_prior"], 
                                                          MFMB4_mf_mb_agent_params["use_orig"], MFMB4_mf_mb_agent_params["use_p"], 
                                                          MFMB4_mf_mb_agent_params["restrict_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                                                          MFMB4_mf_mb_agent_params["min_alpha"], 
                                                          BCC6_p_r_c_MFMB4_mf_mb_base_dir, global_experiment_parameters, BCC6_p_r_c_data["valid"], remove_old=False)


BCC_6pars_planning_repetition_weight_cached_cross_fitting_MFMB_4pars_mf_mb
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_MFMB4_mf_mb_WAIC = os.path.join(BCC6_p_r_c_MFMB4_mf_mb_base_dir, BCC6_p_r_c_MFMB4_mf_mb_fname_base+"_WAIC.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_MFMB4_mf_mb_WAIC = iu.calculate_waic(BCC6_p_r_c_data, BCC6_p_r_c_MFMB4_mf_mb_agent, BCC6_p_r_c_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC6_p_r_c_MFMB4_mf_mb_WAIC)
    with open(fname_BCC6_p_r_c_MFMB4_mf_mb_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC6_p_r_c_MFMB4_mf_mb_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC6_p_r_c_MFMB4_mf_mb_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_MFMB4_mf_mb_log_like = os.path.join(BCC6_p_r_c_MFMB4_mf_mb_base_dir, BCC6_p_r_c_MFMB4_mf_mb_fname_base+"_log_likelihood.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_MFMB4_mf_mb_log_like = iu.calculate_log_likelihood(BCC6_p_r_c_data, BCC6_p_r_c_MFMB4_mf_mb_agent, BCC6_p_r_c_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC6_p_r_c_MFMB4_mf_mb_log_like)
    with open(fname_BCC6_p_r_c_MFMB4_mf_mb_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC6_p_r_c_MFMB4_mf_mb_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC6_p_r_c_MFMB4_mf_mb_log_like = jpickle.decode(pickled_log_like)


## 4.6 MFMB4 MF MB prior cached inference

In [ ]:
BCC6_p_r_c_MFMB6_mf_mb_prior_fname_base = BCC6_p_r_c_agent_type+"_cross_fitting_"+MFMB6_mf_mb_prior_agent_type
print(BCC6_p_r_c_MFMB6_mf_mb_prior_fname_base)
# define folder where we want to save data
BCC6_p_r_c_MFMB6_mf_mb_prior_base_dir = os.path.join(cross_fitting_folder,BCC6_p_r_c_MFMB6_mf_mb_prior_fname_base[:-1])

BCC6_p_r_c_MFMB6_mf_mb_prior_mean_df, BCC6_p_r_c_MFMB6_mf_mb_prior_sample_df, BCC6_p_r_c_MFMB6_mf_mb_prior_locs_df = \
    load_MFMB_results(MFMB6_mf_mb_prior_agent_params["learn_prior"], MFMB6_mf_mb_prior_agent_params["use_orig"], 
                      MFMB6_mf_mb_prior_agent_params["use_p"], MFMB6_mf_mb_prior_agent_params["restrict_alpha"], 
                      MFMB6_mf_mb_prior_agent_params["min_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                      BCC6_p_r_c_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, BCC6_p_r_c_data["valid"], 
                      BCC6_p_r_c_MFMB6_mf_mb_prior_fname_base, num_steps, 
                      BCC6_p_r_c_data, MFMB6_mf_mb_prior_agent_params["param_names"])

BCC6_p_r_c_MFMB6_mf_mb_prior_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB6_mf_mb_prior_agent_params["learn_prior"], 
                                                          MFMB6_mf_mb_prior_agent_params["use_orig"], MFMB6_mf_mb_prior_agent_params["use_p"], 
                                                          MFMB6_mf_mb_prior_agent_params["restrict_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                                                          MFMB6_mf_mb_prior_agent_params["min_alpha"], 
                                                          BCC6_p_r_c_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, BCC6_p_r_c_data["valid"], remove_old=False)


BCC_6pars_planning_repetition_weight_cached_cross_fitting_MFMB_6pars_mf_mb_prior
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_MFMB6_mf_mb_prior_WAIC = os.path.join(BCC6_p_r_c_MFMB6_mf_mb_prior_base_dir, BCC6_p_r_c_MFMB6_mf_mb_prior_fname_base+"_WAIC.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_MFMB6_mf_mb_prior_WAIC = iu.calculate_waic(BCC6_p_r_c_data, BCC6_p_r_c_MFMB6_mf_mb_prior_agent, BCC6_p_r_c_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(BCC6_p_r_c_MFMB6_mf_mb_prior_WAIC)
    with open(fname_BCC6_p_r_c_MFMB6_mf_mb_prior_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_BCC6_p_r_c_MFMB6_mf_mb_prior_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    BCC6_p_r_c_MFMB6_mf_mb_prior_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_BCC6_p_r_c_MFMB6_mf_mb_prior_log_like = os.path.join(BCC6_p_r_c_MFMB6_mf_mb_prior_base_dir, BCC6_p_r_c_MFMB6_mf_mb_prior_fname_base+"_log_likelihood.json")

if recalc_measures_BCC6_p_r_c:
    BCC6_p_r_c_MFMB6_mf_mb_prior_log_like = iu.calculate_log_likelihood(BCC6_p_r_c_data, BCC6_p_r_c_MFMB6_mf_mb_prior_agent, BCC6_p_r_c_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(BCC6_p_r_c_MFMB6_mf_mb_prior_log_like)
    with open(fname_BCC6_p_r_c_MFMB6_mf_mb_prior_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_BCC6_p_r_c_MFMB6_mf_mb_prior_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    BCC6_p_r_c_MFMB6_mf_mb_prior_log_like = jpickle.decode(pickled_log_like)


# 5. True agent = MFMB4 MF MB

## 5.1 BCC2 inference

In [ ]:
MFMB4_mf_mb_BCC2_p_fname_base = MFMB4_mf_mb_agent_type+"_cross_fitting_"+BCC2_p_agent_type
print(MFMB4_mf_mb_BCC2_p_fname_base)
# define folder where we want to save data
MFMB4_mf_mb_BCC2_p_base_dir = os.path.join(cross_fitting_folder,MFMB4_mf_mb_BCC2_p_fname_base[:-1])

MFMB4_mf_mb_BCC2_p_mean_df, MFMB4_mf_mb_BCC2_p_sample_df, MFMB4_mf_mb_BCC2_p_locs_df = \
    load_BCC_results(BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                     BCC2_p_agent_params["learn_cached"], BCC2_p_agent_params["use_h"],
                     MFMB4_mf_mb_BCC2_p_base_dir, global_experiment_parameters, exp_mask, MFMB4_mf_mb_BCC2_p_fname_base, 
                     num_steps, MFMB4_mf_mb_data, BCC2_p_agent_params["param_names"])

MFMB4_mf_mb_BCC2_p_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                                                    BCC2_p_agent_params["learn_cached"], MFMB4_mf_mb_BCC2_p_base_dir, global_experiment_parameters, 
                                                    MFMB4_mf_mb_data["valid"], remove_old=False, use_h=BCC2_p_agent_params["use_h"])

MFMB_4pars_mf_mb_cross_fitting_BCC_2pars_planning
2
analyzing 188 data sets
2


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_BCC2_p_WAIC = os.path.join(MFMB4_mf_mb_BCC2_p_base_dir, MFMB4_mf_mb_BCC2_p_fname_base+"_WAIC.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_BCC2_p_WAIC = iu.calculate_waic(MFMB4_mf_mb_data, MFMB4_mf_mb_BCC2_p_agent, MFMB4_mf_mb_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB4_mf_mb_BCC2_p_WAIC)
    with open(fname_MFMB4_mf_mb_BCC2_p_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_BCC2_p_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB4_mf_mb_BCC2_p_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_BCC2_p_log_like = os.path.join(MFMB4_mf_mb_BCC2_p_base_dir, MFMB4_mf_mb_BCC2_p_fname_base+"_log_likelihood.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_BCC2_p_log_like = iu.calculate_log_likelihood(MFMB4_mf_mb_data, MFMB4_mf_mb_BCC2_p_agent, MFMB4_mf_mb_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB4_mf_mb_BCC2_p_log_like)
    with open(fname_MFMB4_mf_mb_BCC2_p_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_BCC2_p_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB4_mf_mb_BCC2_p_log_like = jpickle.decode(pickled_log_like)


## 5.2 BCC4 planning repetition inference

In [ ]:
MFMB4_mf_mb_BCC4_p_r_fname_base = MFMB4_mf_mb_agent_type+"_cross_fitting_"+BCC4_p_r_agent_type
print(MFMB4_mf_mb_BCC4_p_r_fname_base)
# define folder where we want to save data
MFMB4_mf_mb_BCC4_p_r_base_dir = os.path.join(cross_fitting_folder,MFMB4_mf_mb_BCC4_p_r_fname_base[:-1])

MFMB4_mf_mb_BCC4_p_r_mean_df, MFMB4_mf_mb_BCC4_p_r_sample_df, MFMB4_mf_mb_BCC4_p_r_locs_df = \
    load_BCC_results(BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                     BCC4_p_r_agent_params["learn_cached"], BCC4_p_r_agent_params["use_h"],
                     MFMB4_mf_mb_BCC4_p_r_base_dir, global_experiment_parameters, exp_mask, MFMB4_mf_mb_BCC4_p_r_fname_base, 
                     num_steps, MFMB4_mf_mb_data, BCC4_p_r_agent_params["param_names"])

MFMB4_mf_mb_BCC4_p_r_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                                                    BCC4_p_r_agent_params["learn_cached"], MFMB4_mf_mb_BCC4_p_r_base_dir, global_experiment_parameters, 
                                                    MFMB4_mf_mb_data["valid"], remove_old=False, use_h=BCC4_p_r_agent_params["use_h"])


MFMB_4pars_mf_mb_cross_fitting_BCC_4pars_planning_repetition_weight
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_BCC4_p_r_WAIC = os.path.join(MFMB4_mf_mb_BCC4_p_r_base_dir, MFMB4_mf_mb_BCC4_p_r_fname_base+"_WAIC.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_BCC4_p_r_WAIC = iu.calculate_waic(MFMB4_mf_mb_data, MFMB4_mf_mb_BCC4_p_r_agent, MFMB4_mf_mb_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB4_mf_mb_BCC4_p_r_WAIC)
    with open(fname_MFMB4_mf_mb_BCC4_p_r_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_BCC4_p_r_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB4_mf_mb_BCC4_p_r_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_BCC4_p_r_log_like = os.path.join(MFMB4_mf_mb_BCC4_p_r_base_dir, MFMB4_mf_mb_BCC4_p_r_fname_base+"_log_likelihood.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_BCC4_p_r_log_like = iu.calculate_log_likelihood(MFMB4_mf_mb_data, MFMB4_mf_mb_BCC4_p_r_agent, MFMB4_mf_mb_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB4_mf_mb_BCC4_p_r_log_like)
    with open(fname_MFMB4_mf_mb_BCC4_p_r_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_BCC4_p_r_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB4_mf_mb_BCC4_p_r_log_like = jpickle.decode(pickled_log_like)


## 5.3 BCC4 planning cached inference

In [ ]:
MFMB4_mf_mb_BCC4_p_c_fname_base = MFMB4_mf_mb_agent_type+"_cross_fitting_"+BCC4_p_c_agent_type
print(MFMB4_mf_mb_BCC4_p_c_fname_base)
# define folder where we want to save data
MFMB4_mf_mb_BCC4_p_c_base_dir = os.path.join(cross_fitting_folder,MFMB4_mf_mb_BCC4_p_c_fname_base[:-1])

MFMB4_mf_mb_BCC4_p_c_mean_df, MFMB4_mf_mb_BCC4_p_c_sample_df, MFMB4_mf_mb_BCC4_p_c_locs_df = \
    load_BCC_results(BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                     BCC4_p_c_agent_params["learn_cached"], BCC4_p_c_agent_params["use_h"],
                     MFMB4_mf_mb_BCC4_p_c_base_dir, global_experiment_parameters, exp_mask, MFMB4_mf_mb_BCC4_p_c_fname_base, 
                     num_steps, MFMB4_mf_mb_data, BCC4_p_c_agent_params["param_names"])

MFMB4_mf_mb_BCC4_p_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                                                    BCC4_p_c_agent_params["learn_cached"], MFMB4_mf_mb_BCC4_p_c_base_dir, global_experiment_parameters, 
                                                    MFMB4_mf_mb_data["valid"], remove_old=False, use_h=BCC4_p_c_agent_params["use_h"])


MFMB_4pars_mf_mb_cross_fitting_BCC_4pars_planning_cached
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_BCC4_p_c_WAIC = os.path.join(MFMB4_mf_mb_BCC4_p_c_base_dir, MFMB4_mf_mb_BCC4_p_c_fname_base+"_WAIC.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_BCC4_p_c_WAIC = iu.calculate_waic(MFMB4_mf_mb_data, MFMB4_mf_mb_BCC4_p_c_agent, MFMB4_mf_mb_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB4_mf_mb_BCC4_p_c_WAIC)
    with open(fname_MFMB4_mf_mb_BCC4_p_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_BCC4_p_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB4_mf_mb_BCC4_p_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_BCC4_p_c_log_like = os.path.join(MFMB4_mf_mb_BCC4_p_c_base_dir, MFMB4_mf_mb_BCC4_p_c_fname_base+"_log_likelihood.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_BCC4_p_c_log_like = iu.calculate_log_likelihood(MFMB4_mf_mb_data, MFMB4_mf_mb_BCC4_p_c_agent, MFMB4_mf_mb_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB4_mf_mb_BCC4_p_c_log_like)
    with open(fname_MFMB4_mf_mb_BCC4_p_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_BCC4_p_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB4_mf_mb_BCC4_p_c_log_like = jpickle.decode(pickled_log_like)


## 5.4 BCC6 planning repetition cached inference

In [ ]:
MFMB4_mf_mb_BCC6_p_r_c_fname_base = MFMB4_mf_mb_agent_type+"_cross_fitting_"+BCC6_p_r_c_agent_type
print(MFMB4_mf_mb_BCC6_p_r_c_fname_base)
# define folder where we want to save data
MFMB4_mf_mb_BCC6_p_r_c_base_dir = os.path.join(cross_fitting_folder,MFMB4_mf_mb_BCC6_p_r_c_fname_base[:-1])

MFMB4_mf_mb_BCC6_p_r_c_mean_df, MFMB4_mf_mb_BCC6_p_r_c_sample_df, MFMB4_mf_mb_BCC6_p_r_c_locs_df = \
    load_BCC_results(BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                     BCC6_p_r_c_agent_params["learn_cached"], BCC6_p_r_c_agent_params["use_h"],
                     MFMB4_mf_mb_BCC6_p_r_c_base_dir, global_experiment_parameters, exp_mask, MFMB4_mf_mb_BCC6_p_r_c_fname_base, 
                     num_steps, MFMB4_mf_mb_data, BCC6_p_r_c_agent_params["param_names"])

MFMB4_mf_mb_BCC6_p_r_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                                                    BCC6_p_r_c_agent_params["learn_cached"], MFMB4_mf_mb_BCC6_p_r_c_base_dir, global_experiment_parameters, 
                                                    MFMB4_mf_mb_data["valid"], remove_old=False, use_h=BCC6_p_r_c_agent_params["use_h"])


MFMB_4pars_mf_mb_cross_fitting_BCC_6pars_planning_repetition_weight_cached
6
analyzing 188 data sets
6


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_BCC6_p_r_c_WAIC = os.path.join(MFMB4_mf_mb_BCC6_p_r_c_base_dir, MFMB4_mf_mb_BCC6_p_r_c_fname_base+"_WAIC.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_BCC6_p_r_c_WAIC = iu.calculate_waic(MFMB4_mf_mb_data, MFMB4_mf_mb_BCC6_p_r_c_agent, MFMB4_mf_mb_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB4_mf_mb_BCC6_p_r_c_WAIC)
    with open(fname_MFMB4_mf_mb_BCC6_p_r_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_BCC6_p_r_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB4_mf_mb_BCC6_p_r_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_BCC6_p_r_c_log_like = os.path.join(MFMB4_mf_mb_BCC6_p_r_c_base_dir, MFMB4_mf_mb_BCC6_p_r_c_fname_base+"_log_like.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_BCC6_p_r_c_log_like = iu.calculate_log_likelihood(MFMB4_mf_mb_data, MFMB4_mf_mb_BCC6_p_r_c_agent, MFMB4_mf_mb_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB4_mf_mb_BCC6_p_r_c_log_like)
    with open(fname_MFMB4_mf_mb_BCC6_p_r_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_BCC6_p_r_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB4_mf_mb_BCC6_p_r_c_log_like = jpickle.decode(pickled_log_like)


## 5.5 MFMB4 MF MB cached inference

In [ ]:
MFMB4_mf_mb_MFMB4_mf_mb_fname_base = MFMB4_mf_mb_agent_type+"_recovery_"
print(MFMB4_mf_mb_MFMB4_mf_mb_fname_base)
# define folder where we want to save data
MFMB4_mf_mb_MFMB4_mf_mb_base_dir = os.path.join(recovery_folder,MFMB4_mf_mb_MFMB4_mf_mb_fname_base[:-1])

MFMB4_mf_mb_MFMB4_mf_mb_mean_df, MFMB4_mf_mb_MFMB4_mf_mb_sample_df, MFMB4_mf_mb_MFMB4_mf_mb_locs_df = \
    load_MFMB_results(MFMB4_mf_mb_agent_params["learn_prior"], MFMB4_mf_mb_agent_params["use_orig"], 
                      MFMB4_mf_mb_agent_params["use_p"], MFMB4_mf_mb_agent_params["restrict_alpha"], 
                      MFMB4_mf_mb_agent_params["min_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                      MFMB4_mf_mb_MFMB4_mf_mb_base_dir, global_experiment_parameters, MFMB4_mf_mb_data["valid"], 
                      MFMB4_mf_mb_MFMB4_mf_mb_fname_base, num_steps, 
                      MFMB4_mf_mb_data, MFMB4_mf_mb_agent_params["param_names"])

MFMB4_mf_mb_MFMB4_mf_mb_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB4_mf_mb_agent_params["learn_prior"], 
                                                          MFMB4_mf_mb_agent_params["use_orig"], MFMB4_mf_mb_agent_params["use_p"], 
                                                          MFMB4_mf_mb_agent_params["restrict_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                                                          MFMB4_mf_mb_agent_params["min_alpha"], 
                                                          MFMB4_mf_mb_MFMB4_mf_mb_base_dir, global_experiment_parameters, MFMB4_mf_mb_data["valid"], remove_old=False)


MFMB_4pars_mf_mb_recovery_
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_MFMB4_mf_mb_WAIC = os.path.join(MFMB4_mf_mb_MFMB4_mf_mb_base_dir, MFMB4_mf_mb_MFMB4_mf_mb_fname_base+"_WAIC.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_MFMB4_mf_mb_WAIC = iu.calculate_waic(MFMB4_mf_mb_data, MFMB4_mf_mb_MFMB4_mf_mb_agent, MFMB4_mf_mb_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB4_mf_mb_MFMB4_mf_mb_WAIC)
    with open(fname_MFMB4_mf_mb_MFMB4_mf_mb_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_MFMB4_mf_mb_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB4_mf_mb_MFMB4_mf_mb_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_MFMB4_mf_mb_log_like = os.path.join(MFMB4_mf_mb_MFMB4_mf_mb_base_dir, MFMB4_mf_mb_MFMB4_mf_mb_fname_base+"_log_likelihood.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_MFMB4_mf_mb_log_like = iu.calculate_log_likelihood(MFMB4_mf_mb_data, MFMB4_mf_mb_MFMB4_mf_mb_agent, MFMB4_mf_mb_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB4_mf_mb_MFMB4_mf_mb_log_like)
    with open(fname_MFMB4_mf_mb_MFMB4_mf_mb_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_MFMB4_mf_mb_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB4_mf_mb_MFMB4_mf_mb_log_like = jpickle.decode(pickled_log_like)


## 5.6 MFMB4 MF MB prior cached inference

In [ ]:
MFMB4_mf_mb_MFMB6_mf_mb_prior_fname_base = MFMB4_mf_mb_agent_type+"_cross_fitting_"+MFMB6_mf_mb_prior_agent_type
print(MFMB4_mf_mb_MFMB6_mf_mb_prior_fname_base)
# define folder where we want to save data
MFMB4_mf_mb_MFMB6_mf_mb_prior_base_dir = os.path.join(cross_fitting_folder,MFMB4_mf_mb_MFMB6_mf_mb_prior_fname_base[:-1])

MFMB4_mf_mb_MFMB6_mf_mb_prior_mean_df, MFMB4_mf_mb_MFMB6_mf_mb_prior_sample_df, MFMB4_mf_mb_MFMB6_mf_mb_prior_locs_df = \
    load_MFMB_results(MFMB6_mf_mb_prior_agent_params["learn_prior"], MFMB6_mf_mb_prior_agent_params["use_orig"], 
                      MFMB6_mf_mb_prior_agent_params["use_p"], MFMB6_mf_mb_prior_agent_params["restrict_alpha"], 
                      MFMB6_mf_mb_prior_agent_params["min_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                      MFMB4_mf_mb_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, MFMB4_mf_mb_data["valid"], 
                      MFMB4_mf_mb_MFMB6_mf_mb_prior_fname_base, num_steps, 
                      MFMB4_mf_mb_data, MFMB6_mf_mb_prior_agent_params["param_names"])

MFMB4_mf_mb_MFMB6_mf_mb_prior_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB6_mf_mb_prior_agent_params["learn_prior"], 
                                                          MFMB6_mf_mb_prior_agent_params["use_orig"], MFMB6_mf_mb_prior_agent_params["use_p"], 
                                                          MFMB6_mf_mb_prior_agent_params["restrict_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                                                          MFMB6_mf_mb_prior_agent_params["min_alpha"], 
                                                          MFMB4_mf_mb_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, MFMB4_mf_mb_data["valid"], remove_old=False)


MFMB_4pars_mf_mb_cross_fitting_MFMB_6pars_mf_mb_prior
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_MFMB6_mf_mb_prior_WAIC = os.path.join(MFMB4_mf_mb_MFMB6_mf_mb_prior_base_dir, MFMB4_mf_mb_MFMB6_mf_mb_prior_fname_base+"_WAIC.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_MFMB6_mf_mb_prior_WAIC = iu.calculate_waic(MFMB4_mf_mb_data, MFMB4_mf_mb_MFMB6_mf_mb_prior_agent, MFMB4_mf_mb_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB4_mf_mb_MFMB6_mf_mb_prior_WAIC)
    with open(fname_MFMB4_mf_mb_MFMB6_mf_mb_prior_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_MFMB6_mf_mb_prior_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB4_mf_mb_MFMB6_mf_mb_prior_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB4_mf_mb_MFMB6_mf_mb_prior_log_like = os.path.join(MFMB4_mf_mb_MFMB6_mf_mb_prior_base_dir, MFMB4_mf_mb_MFMB6_mf_mb_prior_fname_base+"_log_likelihood.json")

if recalc_measures_MFMB4_mf_mb:
    MFMB4_mf_mb_MFMB6_mf_mb_prior_log_like = iu.calculate_log_likelihood(MFMB4_mf_mb_data, MFMB4_mf_mb_MFMB6_mf_mb_prior_agent, MFMB4_mf_mb_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB4_mf_mb_MFMB6_mf_mb_prior_log_like)
    with open(fname_MFMB4_mf_mb_MFMB6_mf_mb_prior_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB4_mf_mb_MFMB6_mf_mb_prior_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB4_mf_mb_MFMB6_mf_mb_prior_log_like = jpickle.decode(pickled_log_like)


# 6. True agent = MFMB6 MF MB prior

## 6.1 BCC2 inference

In [ ]:
MFMB6_mf_mb_prior_BCC2_p_fname_base = MFMB6_mf_mb_prior_agent_type+"_cross_fitting_"+BCC2_p_agent_type
print(MFMB6_mf_mb_prior_BCC2_p_fname_base)
# define folder where we want to save data
MFMB6_mf_mb_prior_BCC2_p_base_dir = os.path.join(cross_fitting_folder,MFMB6_mf_mb_prior_BCC2_p_fname_base[:-1])

MFMB6_mf_mb_prior_BCC2_p_mean_df, MFMB6_mf_mb_prior_BCC2_p_sample_df, MFMB6_mf_mb_prior_BCC2_p_locs_df = \
    load_BCC_results(BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                     BCC2_p_agent_params["learn_cached"], BCC2_p_agent_params["use_h"],
                     MFMB6_mf_mb_prior_BCC2_p_base_dir, global_experiment_parameters, exp_mask, MFMB6_mf_mb_prior_BCC2_p_fname_base, 
                     num_steps, MFMB6_mf_mb_prior_data, BCC2_p_agent_params["param_names"])

MFMB6_mf_mb_prior_BCC2_p_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC2_p_agent_params["learn_rewards"], BCC2_p_agent_params["learn_habit"], 
                                                    BCC2_p_agent_params["learn_cached"], MFMB6_mf_mb_prior_BCC2_p_base_dir, global_experiment_parameters, 
                                                    MFMB6_mf_mb_prior_data["valid"], remove_old=False, use_h=BCC2_p_agent_params["use_h"])

MFMB_6pars_mf_mb_prior_cross_fitting_BCC_2pars_planning
2
analyzing 188 data sets
2


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_BCC2_p_WAIC = os.path.join(MFMB6_mf_mb_prior_BCC2_p_base_dir, MFMB6_mf_mb_prior_BCC2_p_fname_base+"_WAIC.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_BCC2_p_WAIC = iu.calculate_waic(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_BCC2_p_agent, MFMB6_mf_mb_prior_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB6_mf_mb_prior_BCC2_p_WAIC)
    with open(fname_MFMB6_mf_mb_prior_BCC2_p_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_BCC2_p_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB6_mf_mb_prior_BCC2_p_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_BCC2_p_log_like = os.path.join(MFMB6_mf_mb_prior_BCC2_p_base_dir, MFMB6_mf_mb_prior_BCC2_p_fname_base+"_log_likelihood.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_BCC2_p_log_like = iu.calculate_log_likelihood(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_BCC2_p_agent, MFMB6_mf_mb_prior_BCC2_p_locs_df, len(BCC2_p_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB6_mf_mb_prior_BCC2_p_log_like)
    with open(fname_MFMB6_mf_mb_prior_BCC2_p_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_BCC2_p_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB6_mf_mb_prior_BCC2_p_log_like = jpickle.decode(pickled_log_like)


## 6.2 BCC4 planning repetition inference

In [ ]:
MFMB6_mf_mb_prior_BCC4_p_r_fname_base = MFMB6_mf_mb_prior_agent_type+"_cross_fitting_"+BCC4_p_r_agent_type
print(MFMB6_mf_mb_prior_BCC4_p_r_fname_base)
# define folder where we want to save data
MFMB6_mf_mb_prior_BCC4_p_r_base_dir = os.path.join(cross_fitting_folder,MFMB6_mf_mb_prior_BCC4_p_r_fname_base[:-1])

MFMB6_mf_mb_prior_BCC4_p_r_mean_df, MFMB6_mf_mb_prior_BCC4_p_r_sample_df, MFMB6_mf_mb_prior_BCC4_p_r_locs_df = \
    load_BCC_results(BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                     BCC4_p_r_agent_params["learn_cached"], BCC4_p_r_agent_params["use_h"],
                     MFMB6_mf_mb_prior_BCC4_p_r_base_dir, global_experiment_parameters, exp_mask, MFMB6_mf_mb_prior_BCC4_p_r_fname_base, 
                     num_steps, MFMB6_mf_mb_prior_data, BCC4_p_r_agent_params["param_names"])

MFMB6_mf_mb_prior_BCC4_p_r_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_r_agent_params["learn_rewards"], BCC4_p_r_agent_params["learn_habit"], 
                                                    BCC4_p_r_agent_params["learn_cached"], MFMB6_mf_mb_prior_BCC4_p_r_base_dir, global_experiment_parameters, 
                                                    MFMB6_mf_mb_prior_data["valid"], remove_old=False, use_h=BCC4_p_r_agent_params["use_h"])


MFMB_6pars_mf_mb_prior_cross_fitting_BCC_4pars_planning_repetition_weight
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_BCC4_p_r_WAIC = os.path.join(MFMB6_mf_mb_prior_BCC4_p_r_base_dir, MFMB6_mf_mb_prior_BCC4_p_r_fname_base+"_WAIC.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_BCC4_p_r_WAIC = iu.calculate_waic(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_BCC4_p_r_agent, MFMB6_mf_mb_prior_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB6_mf_mb_prior_BCC4_p_r_WAIC)
    with open(fname_MFMB6_mf_mb_prior_BCC4_p_r_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_BCC4_p_r_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB6_mf_mb_prior_BCC4_p_r_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_BCC4_p_r_log_like = os.path.join(MFMB6_mf_mb_prior_BCC4_p_r_base_dir, MFMB6_mf_mb_prior_BCC4_p_r_fname_base+"_log_likelihood.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_BCC4_p_r_log_like = iu.calculate_log_likelihood(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_BCC4_p_r_agent, MFMB6_mf_mb_prior_BCC4_p_r_locs_df, len(BCC4_p_r_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB6_mf_mb_prior_BCC4_p_r_log_like)
    with open(fname_MFMB6_mf_mb_prior_BCC4_p_r_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_BCC4_p_r_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB6_mf_mb_prior_BCC4_p_r_log_like = jpickle.decode(pickled_log_like)


## 6.3 BCC4 planning cached inference

In [ ]:
MFMB6_mf_mb_prior_BCC4_p_c_fname_base = MFMB6_mf_mb_prior_agent_type+"_cross_fitting_"+BCC4_p_c_agent_type
print(MFMB6_mf_mb_prior_BCC4_p_c_fname_base)
# define folder where we want to save data
MFMB6_mf_mb_prior_BCC4_p_c_base_dir = os.path.join(cross_fitting_folder,MFMB6_mf_mb_prior_BCC4_p_c_fname_base[:-1])

MFMB6_mf_mb_prior_BCC4_p_c_mean_df, MFMB6_mf_mb_prior_BCC4_p_c_sample_df, MFMB6_mf_mb_prior_BCC4_p_c_locs_df = \
    load_BCC_results(BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                     BCC4_p_c_agent_params["learn_cached"], BCC4_p_c_agent_params["use_h"],
                     MFMB6_mf_mb_prior_BCC4_p_c_base_dir, global_experiment_parameters, exp_mask, MFMB6_mf_mb_prior_BCC4_p_c_fname_base, 
                     num_steps, MFMB6_mf_mb_prior_data, BCC4_p_c_agent_params["param_names"])

MFMB6_mf_mb_prior_BCC4_p_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC4_p_c_agent_params["learn_rewards"], BCC4_p_c_agent_params["learn_habit"], 
                                                    BCC4_p_c_agent_params["learn_cached"], MFMB6_mf_mb_prior_BCC4_p_c_base_dir, global_experiment_parameters, 
                                                    MFMB6_mf_mb_prior_data["valid"], remove_old=False, use_h=BCC4_p_c_agent_params["use_h"])


MFMB_6pars_mf_mb_prior_cross_fitting_BCC_4pars_planning_cached
4
analyzing 188 data sets
4


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_BCC4_p_c_WAIC = os.path.join(MFMB6_mf_mb_prior_BCC4_p_c_base_dir, MFMB6_mf_mb_prior_BCC4_p_c_fname_base+"_WAIC.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_BCC4_p_c_WAIC = iu.calculate_waic(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_BCC4_p_c_agent, MFMB6_mf_mb_prior_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB6_mf_mb_prior_BCC4_p_c_WAIC)
    with open(fname_MFMB6_mf_mb_prior_BCC4_p_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_BCC4_p_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB6_mf_mb_prior_BCC4_p_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_BCC4_p_c_log_like = os.path.join(MFMB6_mf_mb_prior_BCC4_p_c_base_dir, MFMB6_mf_mb_prior_BCC4_p_c_fname_base+"_log_likelihood.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_BCC4_p_c_log_like = iu.calculate_log_likelihood(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_BCC4_p_c_agent, MFMB6_mf_mb_prior_BCC4_p_c_locs_df, len(BCC4_p_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB6_mf_mb_prior_BCC4_p_c_log_like)
    with open(fname_MFMB6_mf_mb_prior_BCC4_p_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_BCC4_p_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB6_mf_mb_prior_BCC4_p_c_log_like = jpickle.decode(pickled_log_like)


## 6.4 BCC6 planning repetition cached inference

In [ ]:
MFMB6_mf_mb_prior_BCC6_p_r_c_fname_base = MFMB6_mf_mb_prior_agent_type+"_cross_fitting_"+BCC6_p_r_c_agent_type
print(MFMB6_mf_mb_prior_BCC6_p_r_c_fname_base)
# define folder where we want to save data
MFMB6_mf_mb_prior_BCC6_p_r_c_base_dir = os.path.join(cross_fitting_folder,MFMB6_mf_mb_prior_BCC6_p_r_c_fname_base[:-1])

MFMB6_mf_mb_prior_BCC6_p_r_c_mean_df, MFMB6_mf_mb_prior_BCC6_p_r_c_sample_df, MFMB6_mf_mb_prior_BCC6_p_r_c_locs_df = \
    load_BCC_results(BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                     BCC6_p_r_c_agent_params["learn_cached"], BCC6_p_r_c_agent_params["use_h"],
                     MFMB6_mf_mb_prior_BCC6_p_r_c_base_dir, global_experiment_parameters, exp_mask, MFMB6_mf_mb_prior_BCC6_p_r_c_fname_base, 
                     num_steps, MFMB6_mf_mb_prior_data, BCC6_p_r_c_agent_params["param_names"])

MFMB6_mf_mb_prior_BCC6_p_r_c_agent = tu.set_up_Bayesian_inference_agent(n_agents, BCC6_p_r_c_agent_params["learn_rewards"], BCC6_p_r_c_agent_params["learn_habit"], 
                                                    BCC6_p_r_c_agent_params["learn_cached"], MFMB6_mf_mb_prior_BCC6_p_r_c_base_dir, global_experiment_parameters, 
                                                    MFMB6_mf_mb_prior_data["valid"], remove_old=False, use_h=BCC6_p_r_c_agent_params["use_h"])


MFMB_6pars_mf_mb_prior_cross_fitting_BCC_6pars_planning_repetition_weight_cached
6
analyzing 188 data sets
6


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_BCC6_p_r_c_WAIC = os.path.join(MFMB6_mf_mb_prior_BCC6_p_r_c_base_dir, MFMB6_mf_mb_prior_BCC6_p_r_c_fname_base+"_WAIC.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_BCC6_p_r_c_WAIC = iu.calculate_waic(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_BCC6_p_r_c_agent, MFMB6_mf_mb_prior_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB6_mf_mb_prior_BCC6_p_r_c_WAIC)
    with open(fname_MFMB6_mf_mb_prior_BCC6_p_r_c_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_BCC6_p_r_c_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB6_mf_mb_prior_BCC6_p_r_c_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_BCC6_p_r_c_log_like = os.path.join(MFMB6_mf_mb_prior_BCC6_p_r_c_base_dir, MFMB6_mf_mb_prior_BCC6_p_r_c_fname_base+"_log_likelihood.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_BCC6_p_r_c_log_like = iu.calculate_log_likelihood(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_BCC6_p_r_c_agent, MFMB6_mf_mb_prior_BCC6_p_r_c_locs_df, len(BCC6_p_r_c_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB6_mf_mb_prior_BCC6_p_r_c_log_like)
    with open(fname_MFMB6_mf_mb_prior_BCC6_p_r_c_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_BCC6_p_r_c_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB6_mf_mb_prior_BCC6_p_r_c_log_like = jpickle.decode(pickled_log_like)


## 6.5 MFMB4 MF MB cached inference

In [ ]:
MFMB6_mf_mb_prior_MFMB4_mf_mb_fname_base = MFMB6_mf_mb_prior_agent_type+"_cross_fitting_"+MFMB4_mf_mb_agent_type
print(MFMB6_mf_mb_prior_MFMB4_mf_mb_fname_base)
# define folder where we want to save data
MFMB6_mf_mb_prior_MFMB4_mf_mb_base_dir = os.path.join(cross_fitting_folder,MFMB6_mf_mb_prior_MFMB4_mf_mb_fname_base[:-1])

MFMB6_mf_mb_prior_MFMB4_mf_mb_mean_df, MFMB6_mf_mb_prior_MFMB4_mf_mb_sample_df, MFMB6_mf_mb_prior_MFMB4_mf_mb_locs_df = \
    load_MFMB_results(MFMB4_mf_mb_agent_params["learn_prior"], MFMB4_mf_mb_agent_params["use_orig"], 
                      MFMB4_mf_mb_agent_params["use_p"], MFMB4_mf_mb_agent_params["restrict_alpha"], 
                      MFMB4_mf_mb_agent_params["min_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                      MFMB4_mf_mb_MFMB4_mf_mb_base_dir, global_experiment_parameters, MFMB6_mf_mb_prior_data["valid"], 
                      MFMB4_mf_mb_MFMB4_mf_mb_fname_base, num_steps, 
                      MFMB6_mf_mb_prior_data, MFMB4_mf_mb_agent_params["param_names"])

MFMB6_mf_mb_prior_MFMB4_mf_mb_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB4_mf_mb_agent_params["learn_prior"], 
                                                          MFMB4_mf_mb_agent_params["use_orig"], MFMB4_mf_mb_agent_params["use_p"], 
                                                          MFMB4_mf_mb_agent_params["restrict_alpha"], MFMB4_mf_mb_agent_params["max_dt"], 
                                                          MFMB4_mf_mb_agent_params["min_alpha"], 
                                                          MFMB6_mf_mb_prior_MFMB4_mf_mb_base_dir, global_experiment_parameters, MFMB6_mf_mb_prior_data["valid"], remove_old=False)


MFMB_6pars_mf_mb_prior_cross_fitting_MFMB_4pars_mf_mb
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_MFMB4_mf_mb_WAIC = os.path.join(MFMB6_mf_mb_prior_MFMB4_mf_mb_base_dir, MFMB6_mf_mb_prior_MFMB4_mf_mb_fname_base+"_WAIC.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_MFMB4_mf_mb_WAIC = iu.calculate_waic(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_MFMB4_mf_mb_agent, MFMB6_mf_mb_prior_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB6_mf_mb_prior_MFMB4_mf_mb_WAIC)
    with open(fname_MFMB6_mf_mb_prior_MFMB4_mf_mb_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_MFMB4_mf_mb_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB6_mf_mb_prior_MFMB4_mf_mb_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_MFMB4_mf_mb_log_like = os.path.join(MFMB6_mf_mb_prior_MFMB4_mf_mb_base_dir, MFMB6_mf_mb_prior_MFMB4_mf_mb_fname_base+"_log_likelihood.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_MFMB4_mf_mb_log_like = iu.calculate_log_likelihood(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_MFMB4_mf_mb_agent, MFMB6_mf_mb_prior_MFMB4_mf_mb_locs_df, len(MFMB4_mf_mb_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB6_mf_mb_prior_MFMB4_mf_mb_log_like)
    with open(fname_MFMB6_mf_mb_prior_MFMB4_mf_mb_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_MFMB4_mf_mb_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB6_mf_mb_prior_MFMB4_mf_mb_log_like = jpickle.decode(pickled_log_like)


## 6.6 MFMB4 MF MB prior cached inference

In [ ]:

MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_fname_base = MFMB6_mf_mb_prior_agent_type+"_recovery_"
print(MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_fname_base)
# define folder where we want to save data
MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_base_dir = os.path.join(recovery_folder,MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_fname_base[:-1])

MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_mean_df, MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_sample_df, MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_locs_df = \
    load_MFMB_results(MFMB6_mf_mb_prior_agent_params["learn_prior"], MFMB6_mf_mb_prior_agent_params["use_orig"], 
                      MFMB6_mf_mb_prior_agent_params["use_p"], MFMB6_mf_mb_prior_agent_params["restrict_alpha"], 
                      MFMB6_mf_mb_prior_agent_params["min_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                      MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, MFMB6_mf_mb_prior_data["valid"], 
                      MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_fname_base, num_steps, 
                      MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_agent_params["param_names"])

MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_agent = tu.set_up_mbmf_inference_agent(n_agents, MFMB6_mf_mb_prior_agent_params["learn_prior"], 
                                                          MFMB6_mf_mb_prior_agent_params["use_orig"], MFMB6_mf_mb_prior_agent_params["use_p"], 
                                                          MFMB6_mf_mb_prior_agent_params["restrict_alpha"], MFMB6_mf_mb_prior_agent_params["max_dt"], 
                                                          MFMB6_mf_mb_prior_agent_params["min_alpha"], 
                                                          MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_base_dir, global_experiment_parameters, MFMB6_mf_mb_prior_data["valid"], remove_old=False)


MFMB_6pars_mf_mb_prior_recovery_
analyzing 188 data sets


/home/sarah/python_venvs/TwoStageStrategies/lib/python3.13/site-packages/pyro/params/param_store.py:334: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(inp

In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_WAIC = os.path.join(MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_base_dir, MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_fname_base+"_WAIC.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_WAIC = iu.calculate_waic(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_agent, MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"], max_samples=WAIC_max_samples)

    pickled_WAIC = jpickle.encode(MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_WAIC)
    with open(fname_MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_WAIC, 'w') as outfile:
        json.dump(pickled_WAIC, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_WAIC, 'r') as infile:
        pickled_WAIC = json.load(infile)
    MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_WAIC = jpickle.decode(pickled_WAIC)


In [ ]:
# calculate or load model comparison measure

fname_MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_log_like = os.path.join(MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_base_dir, MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_fname_base+"_log_like.json")

if recalc_measures_MFMB6_mf_mb_prior:
    MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_log_like = iu.calculate_log_likelihood(MFMB6_mf_mb_prior_data, MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_agent, MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_locs_df, len(MFMB6_mf_mb_prior_agent_params["param_names"]), 
                            global_experiment_parameters["trials"], global_experiment_parameters["T"])

    pickled_log_like = jpickle.encode(MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_log_like)
    with open(fname_MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_log_like, 'w') as outfile:
        json.dump(pickled_log_like, outfile)
    
else:

    with open(fname_MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_log_like, 'r') as infile:
        pickled_log_like = json.load(infile)
    MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_log_like = jpickle.decode(pickled_log_like)


# Get all WAICs

In [ ]:
BCC2_p_data_all_total_WAICs = torch.stack([BCC2_p_BCC2_p_WAIC, BCC2_p_BCC4_p_r_WAIC, 
                                           BCC2_p_BCC4_p_c_WAIC, BCC2_p_BCC6_p_r_c_WAIC,
                                           BCC2_p_MFMB4_mf_mb_WAIC, BCC2_p_MFMB6_mf_mb_prior_WAIC], dim=-1)

BCC2_p_winning_model = BCC2_p_data_all_total_WAICs.argmin(dim=-1)

print("BCC2 planning wins", (BCC2_p_winning_model==0).sum())
print("BCC4 planning repetition wins", (BCC2_p_winning_model==1).sum())
print("BCC4 planning cached wins", (BCC2_p_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (BCC2_p_winning_model==3).sum())
print("MFMB4 MF MB wins", (BCC2_p_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (BCC2_p_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-BCC2_p_data_all_total_WAICs, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-BCC2_p_data_all_total_WAICs.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-BCC2_p_data_all_total_WAICs)

BCC2 planning wins tensor(171)
BCC4 planning repetition wins tensor(0)
BCC4 planning cached wins tensor(15)
BCC6 planning repetition cached repetition wins tensor(0)
MFMB4 MF MB wins tensor(2)
MFMB6 MF MP Prior wins tensor(0)
torch.Size([188, 6])
tensor([9.0722e-01, 3.2060e-08, 8.4075e-02, 1.8588e-10, 8.7070e-03, 5.7197e-10])
tensor([1., 0., 0., 0., 0., 0.])
p model mean according to measure tensor([9.0722e-01, 3.2060e-08, 8.4075e-02, 1.8588e-10, 8.7070e-03, 5.7197e-10])
best model: tensor(0) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(736.4908472675322), pvalue=np.float64(0.0), df=np.int64(499))


In [ ]:
BCC4_p_r_data_all_total_WAICs = torch.stack([BCC4_p_r_BCC2_p_WAIC, BCC4_p_r_BCC4_p_r_WAIC, 
                                           BCC4_p_r_BCC4_p_c_WAIC, BCC4_p_r_BCC6_p_r_c_WAIC,
                                           BCC4_p_r_MFMB4_mf_mb_WAIC, BCC4_p_r_MFMB6_mf_mb_prior_WAIC], dim=-1)

BCC4_p_r_winning_model = BCC4_p_r_data_all_total_WAICs.argmin(dim=-1)

print("BCC2 planning wins", (BCC4_p_r_winning_model==0).sum())
print("BCC4 planning repetition wins", (BCC4_p_r_winning_model==1).sum())
print("BCC4 planning cached wins", (BCC4_p_r_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (BCC4_p_r_winning_model==3).sum())
print("MFMB4 MF MB wins", (BCC4_p_r_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (BCC4_p_r_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-BCC4_p_r_data_all_total_WAICs, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-BCC4_p_r_data_all_total_WAICs.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-BCC4_p_r_data_all_total_WAICs)

BCC2 planning wins tensor(2)
BCC4 planning repetition wins tensor(160)
BCC4 planning cached wins tensor(0)
BCC6 planning repetition cached repetition wins tensor(17)
MFMB4 MF MB wins tensor(4)
MFMB6 MF MP Prior wins tensor(5)
torch.Size([188, 6])
tensor([1.1284e-02, 8.6956e-01, 2.3982e-06, 7.1012e-02, 2.0192e-02, 2.7949e-02])
tensor([0., 1., 0., 0., 0., 0.])
p model mean according to measure tensor([1.1284e-02, 8.6956e-01, 2.3982e-06, 7.1012e-02, 2.0192e-02, 2.7949e-02])
best model: tensor(1) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(643.416742369612), pvalue=np.float64(0.0), df=np.int64(499))


In [ ]:
BCC4_p_c_data_all_total_WAICs = torch.stack([BCC4_p_c_BCC2_p_WAIC, BCC4_p_c_BCC4_p_r_WAIC, 
                                           BCC4_p_c_BCC4_p_c_WAIC, BCC4_p_c_BCC6_p_r_c_WAIC,
                                           BCC4_p_c_MFMB4_mf_mb_WAIC, BCC4_p_c_MFMB6_mf_mb_prior_WAIC], dim=-1)

BCC4_p_c_winning_model = BCC4_p_c_data_all_total_WAICs.argmin(dim=-1)

print("BCC2 planning wins", (BCC4_p_c_winning_model==0).sum())
print("BCC4 planning repetition wins", (BCC4_p_c_winning_model==1).sum())
print("BCC4 planning cached wins", (BCC4_p_c_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (BCC4_p_c_winning_model==3).sum())
print("MFMB4 MF MB wins", (BCC4_p_c_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (BCC4_p_c_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-BCC4_p_c_data_all_total_WAICs, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-BCC4_p_c_data_all_total_WAICs.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-BCC4_p_c_data_all_total_WAICs)

BCC2 planning wins tensor(18)
BCC4 planning repetition wins tensor(0)
BCC4 planning cached wins tensor(170)
BCC6 planning repetition cached repetition wins tensor(0)
MFMB4 MF MB wins tensor(0)
MFMB6 MF MP Prior wins tensor(0)
torch.Size([188, 6])
tensor([9.3742e-02, 9.9459e-06, 9.0625e-01, 4.0868e-14, 3.5151e-06, 6.8822e-13])
tensor([0., 0., 1., 0., 0., 0.])
p model mean according to measure tensor([9.3742e-02, 9.9459e-06, 9.0625e-01, 4.0868e-14, 3.5151e-06, 6.8822e-13])
best model: tensor(2) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(826.1476083493155), pvalue=np.float64(0.0), df=np.int64(499))


In [ ]:
BCC6_p_r_c_data_all_total_WAICs = torch.stack([BCC6_p_r_c_BCC2_p_WAIC, BCC6_p_r_c_BCC4_p_r_WAIC, 
                                           BCC6_p_r_c_BCC4_p_c_WAIC, BCC6_p_r_c_BCC6_p_r_c_WAIC,
                                           BCC6_p_r_c_MFMB4_mf_mb_WAIC, BCC6_p_r_c_MFMB6_mf_mb_prior_WAIC], dim=-1)

BCC6_p_r_c_winning_model = BCC6_p_r_c_data_all_total_WAICs.argmax(dim=-1)

print("BCC2 planning wins", (BCC6_p_r_c_winning_model==0).sum())
print("BCC4 planning repetition wins", (BCC6_p_r_c_winning_model==1).sum())
print("BCC4 planning cached wins", (BCC6_p_r_c_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (BCC6_p_r_c_winning_model==3).sum())
print("MFMB4 MF MB wins", (BCC6_p_r_c_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (BCC6_p_r_c_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-BCC6_p_r_c_data_all_total_WAICs, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-BCC6_p_r_c_data_all_total_WAICs.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-BCC6_p_r_c_data_all_total_WAICs)

BCC2 planning wins tensor(68)
BCC4 planning repetition wins tensor(11)
BCC4 planning cached wins tensor(102)
BCC6 planning repetition cached repetition wins tensor(0)
MFMB4 MF MB wins tensor(2)
MFMB6 MF MP Prior wins tensor(5)
torch.Size([188, 6])
tensor([4.6287e-06, 5.1363e-02, 1.5962e-02, 8.9506e-01, 1.2233e-02, 2.5374e-02])
tensor([0., 0., 0., 1., 0., 0.])
p model mean according to measure tensor([4.6287e-06, 5.1363e-02, 1.5962e-02, 8.9506e-01, 1.2233e-02, 2.5374e-02])
best model: tensor(3) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(701.2049330330934), pvalue=np.float64(0.0), df=np.int64(499))


In [ ]:
MFMB4_mf_mb_data_all_total_WAICs = torch.stack([MFMB4_mf_mb_BCC2_p_WAIC, MFMB4_mf_mb_BCC4_p_r_WAIC, 
                                           MFMB4_mf_mb_BCC4_p_c_WAIC, MFMB4_mf_mb_BCC6_p_r_c_WAIC,
                                           MFMB4_mf_mb_MFMB4_mf_mb_WAIC, MFMB4_mf_mb_MFMB6_mf_mb_prior_WAIC], dim=-1)

MFMB4_mf_mb_winning_model = MFMB4_mf_mb_data_all_total_WAICs.argmin(dim=-1)

print("BCC2 planning wins", (MFMB4_mf_mb_winning_model==0).sum())
print("BCC4 planning repetition wins", (MFMB4_mf_mb_winning_model==1).sum())
print("BCC4 planning cached wins", (MFMB4_mf_mb_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (MFMB4_mf_mb_winning_model==3).sum())
print("MFMB4 MF MB wins", (MFMB4_mf_mb_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (MFMB4_mf_mb_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-MFMB4_mf_mb_data_all_total_WAICs, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-MFMB4_mf_mb_data_all_total_WAICs.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-MFMB4_mf_mb_data_all_total_WAICs)

BCC2 planning wins tensor(9)
BCC4 planning repetition wins tensor(0)
BCC4 planning cached wins tensor(0)
BCC6 planning repetition cached repetition wins tensor(4)
MFMB4 MF MB wins tensor(141)
MFMB6 MF MP Prior wins tensor(34)
torch.Size([188, 6])
tensor([4.9736e-02, 5.6683e-03, 2.0860e-05, 1.5338e-02, 7.4262e-01, 1.8661e-01])
tensor([0., 0., 0., 0., 1., 0.])
p model mean according to measure tensor([4.9736e-02, 5.6683e-03, 2.0860e-05, 1.5338e-02, 7.4262e-01, 1.8661e-01])
best model: tensor(4) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(404.69483705511647), pvalue=np.float64(0.0), df=np.int64(499))


In [ ]:
MFMB6_mf_mb_prior_data_all_total_WAICs = torch.stack([MFMB6_mf_mb_prior_BCC2_p_WAIC, MFMB6_mf_mb_prior_BCC4_p_r_WAIC, 
                                           MFMB6_mf_mb_prior_BCC4_p_c_WAIC, MFMB6_mf_mb_prior_BCC6_p_r_c_WAIC,
                                           MFMB6_mf_mb_prior_MFMB4_mf_mb_WAIC, MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_WAIC], dim=-1)

MFMB6_mf_mb_prior_winning_model = MFMB6_mf_mb_prior_data_all_total_WAICs.argmin(dim=-1)

print("BCC2 planning wins", (MFMB6_mf_mb_prior_winning_model==0).sum())
print("BCC4 planning repetition wins", (MFMB6_mf_mb_prior_winning_model==1).sum())
print("BCC4 planning cached wins", (MFMB6_mf_mb_prior_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (MFMB6_mf_mb_prior_winning_model==3).sum())
print("MFMB4 MF MB wins", (MFMB6_mf_mb_prior_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (MFMB6_mf_mb_prior_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-MFMB6_mf_mb_prior_data_all_total_WAICs, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-MFMB6_mf_mb_prior_data_all_total_WAICs.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-MFMB6_mf_mb_prior_data_all_total_WAICs)

BCC2 planning wins tensor(0)
BCC4 planning repetition wins tensor(0)
BCC4 planning cached wins tensor(1)
BCC6 planning repetition cached repetition wins tensor(1)
MFMB4 MF MB wins tensor(6)
MFMB6 MF MP Prior wins tensor(180)
torch.Size([188, 6])
tensor([1.4312e-04, 1.0512e-03, 5.1761e-03, 3.4378e-03, 3.2430e-02, 9.5776e-01])
tensor([0., 0., 0., 0., 0., 1.])
p model mean according to measure tensor([1.4312e-04, 1.0512e-03, 5.1761e-03, 3.4378e-03, 3.2430e-02, 9.5776e-01])
best model: tensor(5) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(1168.214798190931), pvalue=np.float64(0.0), df=np.int64(499))


# Get all Log Likelihoodss

In [ ]:
BCC2_p_data_all_total_log_likes = torch.stack([BCC2_p_BCC2_p_log_like, BCC2_p_BCC4_p_r_log_like, 
                                           BCC2_p_BCC4_p_c_log_like, BCC2_p_BCC6_p_r_c_log_like,
                                           BCC2_p_MFMB4_mf_mb_log_like, BCC2_p_MFMB6_mf_mb_prior_log_like], dim=-1)

BCC2_p_winning_model = BCC2_p_data_all_total_log_likes.argmin(dim=-1)

print("BCC2 planning wins", (BCC2_p_winning_model==0).sum())
print("BCC4 planning repetition wins", (BCC2_p_winning_model==1).sum())
print("BCC4 planning cached wins", (BCC2_p_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (BCC2_p_winning_model==3).sum())
print("MFMB4 MF MB wins", (BCC2_p_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (BCC2_p_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-BCC2_p_data_all_total_log_likes, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-BCC2_p_data_all_total_log_likes.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-BCC2_p_data_all_total_log_likes)

BCC2 planning wins tensor(166)
BCC4 planning repetition wins tensor(3)
BCC4 planning cached wins tensor(16)
BCC6 planning repetition cached repetition wins tensor(1)
MFMB4 MF MB wins tensor(2)
MFMB6 MF MP Prior wins tensor(0)
torch.Size([188, 6])
tensor([0.7859, 0.0243, 0.1617, 0.0115, 0.0127, 0.0039])
tensor([1., 0., 0., 0., 0., 0.])
p model mean according to measure tensor([0.7859, 0.0243, 0.1617, 0.0115, 0.0127, 0.0039])
best model: tensor(0) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(460.1495013426043), pvalue=np.float64(0.0), df=np.int64(499))


In [ ]:
BCC4_p_r_data_all_total_log_likes = torch.stack([BCC4_p_r_BCC2_p_log_like, BCC4_p_r_BCC4_p_r_log_like, 
                                           BCC4_p_r_BCC4_p_c_log_like, BCC4_p_r_BCC6_p_r_c_log_like,
                                           BCC4_p_r_MFMB4_mf_mb_log_like, BCC4_p_r_MFMB6_mf_mb_prior_log_like], dim=-1)

BCC4_p_r_winning_model = BCC4_p_r_data_all_total_log_likes.argmin(dim=-1)

print("BCC2 planning wins", (BCC4_p_r_winning_model==0).sum())
print("BCC4 planning repetition wins", (BCC4_p_r_winning_model==1).sum())
print("BCC4 planning cached wins", (BCC4_p_r_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (BCC4_p_r_winning_model==3).sum())
print("MFMB4 MF MB wins", (BCC4_p_r_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (BCC4_p_r_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-BCC4_p_r_data_all_total_log_likes, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-BCC4_p_r_data_all_total_log_likes.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-BCC4_p_r_data_all_total_log_likes)

BCC2 planning wins tensor(0)
BCC4 planning repetition wins tensor(164)
BCC4 planning cached wins tensor(0)
BCC6 planning repetition cached repetition wins tensor(23)
MFMB4 MF MB wins tensor(0)
MFMB6 MF MP Prior wins tensor(1)
torch.Size([188, 6])
tensor([3.1670e-05, 8.0187e-01, 2.6424e-06, 1.6499e-01, 1.0998e-02, 2.2110e-02])
tensor([0., 1., 0., 0., 0., 0.])
p model mean according to measure tensor([3.1670e-05, 8.0187e-01, 2.6424e-06, 1.6499e-01, 1.0998e-02, 2.2110e-02])
best model: tensor(1) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(483.63029710330557), pvalue=np.float64(0.0), df=np.int64(499))


In [ ]:
BCC4_p_c_data_all_total_log_likes = torch.stack([BCC4_p_c_BCC2_p_log_like, BCC4_p_c_BCC4_p_r_log_like, 
                                           BCC4_p_c_BCC4_p_c_log_like, BCC4_p_c_BCC6_p_r_c_log_like,
                                           BCC4_p_c_MFMB4_mf_mb_log_like, BCC4_p_c_MFMB6_mf_mb_prior_log_like], dim=-1)

BCC4_p_c_winning_model = BCC4_p_c_data_all_total_log_likes.argmin(dim=-1)

print("BCC2 planning wins", (BCC4_p_c_winning_model==0).sum())
print("BCC4 planning repetition wins", (BCC4_p_c_winning_model==1).sum())
print("BCC4 planning cached wins", (BCC4_p_c_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (BCC4_p_c_winning_model==3).sum())
print("MFMB4 MF MB wins", (BCC4_p_c_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (BCC4_p_c_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-BCC4_p_c_data_all_total_log_likes, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-BCC4_p_c_data_all_total_log_likes.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-BCC4_p_c_data_all_total_log_likes)

BCC2 planning wins tensor(1)
BCC4 planning repetition wins tensor(0)
BCC4 planning cached wins tensor(187)
BCC6 planning repetition cached repetition wins tensor(0)
MFMB4 MF MB wins tensor(0)
MFMB6 MF MP Prior wins tensor(0)
torch.Size([188, 6])
tensor([8.4529e-03, 8.6824e-04, 9.8727e-01, 3.3964e-03, 1.2435e-05, 4.1733e-06])
tensor([0., 0., 1., 0., 0., 0.])
p model mean according to measure tensor([8.4529e-03, 8.6824e-04, 9.8727e-01, 3.3964e-03, 1.2435e-05, 4.1733e-06])
best model: tensor(2) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(2186.919021047204), pvalue=np.float64(0.0), df=np.int64(499))


In [ ]:
BCC6_p_r_c_data_all_total_log_likes = torch.stack([BCC6_p_r_c_BCC2_p_log_like, BCC6_p_r_c_BCC4_p_r_log_like, 
                                           BCC6_p_r_c_BCC4_p_c_log_like, BCC6_p_r_c_BCC6_p_r_c_log_like,
                                           BCC6_p_r_c_MFMB4_mf_mb_log_like, BCC6_p_r_c_MFMB6_mf_mb_prior_log_like], dim=-1)

BCC6_p_r_c_winning_model = BCC6_p_r_c_data_all_total_log_likes.argmin(dim=-1)

print("BCC2 planning wins", (BCC6_p_r_c_winning_model==0).sum())
print("BCC4 planning repetition wins", (BCC6_p_r_c_winning_model==1).sum())
print("BCC4 planning cached wins", (BCC6_p_r_c_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (BCC6_p_r_c_winning_model==3).sum())
print("MFMB4 MF MB wins", (BCC6_p_r_c_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (BCC6_p_r_c_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-BCC6_p_r_c_data_all_total_log_likes, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-BCC6_p_r_c_data_all_total_log_likes.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-BCC6_p_r_c_data_all_total_log_likes)

BCC2 planning wins tensor(0)
BCC4 planning repetition wins tensor(13)
BCC4 planning cached wins tensor(0)
BCC6 planning repetition cached repetition wins tensor(173)
MFMB4 MF MB wins tensor(0)
MFMB6 MF MP Prior wins tensor(2)
torch.Size([188, 6])
tensor([1.3498e-09, 9.8556e-02, 2.0940e-05, 8.7149e-01, 1.0842e-02, 1.9087e-02])
tensor([0., 0., 0., 1., 0., 0.])
p model mean according to measure tensor([1.3498e-09, 9.8556e-02, 2.0940e-05, 8.7149e-01, 1.0842e-02, 1.9087e-02])
best model: tensor(3) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(644.1721593898383), pvalue=np.float64(0.0), df=np.int64(499))


In [ ]:
MFMB4_mf_mb_data_all_total_log_likes = torch.stack([MFMB4_mf_mb_BCC2_p_log_like, MFMB4_mf_mb_BCC4_p_r_log_like, 
                                           MFMB4_mf_mb_BCC4_p_c_log_like, MFMB4_mf_mb_BCC6_p_r_c_log_like,
                                           MFMB4_mf_mb_MFMB4_mf_mb_log_like, MFMB4_mf_mb_MFMB6_mf_mb_prior_log_like], dim=-1)

MFMB4_mf_mb_winning_model = MFMB4_mf_mb_data_all_total_log_likes.argmin(dim=-1)

print("BCC2 planning wins", (MFMB4_mf_mb_winning_model==0).sum())
print("BCC4 planning repetition wins", (MFMB4_mf_mb_winning_model==1).sum())
print("BCC4 planning cached wins", (MFMB4_mf_mb_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (MFMB4_mf_mb_winning_model==3).sum())
print("MFMB4 MF MB wins", (MFMB4_mf_mb_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (MFMB4_mf_mb_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-MFMB4_mf_mb_data_all_total_log_likes, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-MFMB4_mf_mb_data_all_total_log_likes.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-MFMB4_mf_mb_data_all_total_log_likes)

BCC2 planning wins tensor(1)
BCC4 planning repetition wins tensor(3)
BCC4 planning cached wins tensor(0)
BCC6 planning repetition cached repetition wins tensor(4)
MFMB4 MF MB wins tensor(131)
MFMB6 MF MP Prior wins tensor(49)
torch.Size([188, 6])
tensor([0.0046, 0.0189, 0.0025, 0.0156, 0.6191, 0.3392])
tensor([0., 0., 0., 0., 1., 0.])
p model mean according to measure tensor([0.0046, 0.0189, 0.0025, 0.0156, 0.6191, 0.3392])
best model: tensor(4) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(287.482416394713), pvalue=np.float64(0.0), df=np.int64(499))


In [ ]:
MFMB6_mf_mb_prior_data_all_total_log_likes = torch.stack([MFMB6_mf_mb_prior_BCC2_p_log_like, MFMB6_mf_mb_prior_BCC4_p_r_log_like, 
                                           MFMB6_mf_mb_prior_BCC4_p_c_log_like, MFMB6_mf_mb_prior_BCC6_p_r_c_log_like,
                                           MFMB6_mf_mb_prior_MFMB4_mf_mb_log_like, MFMB6_mf_mb_prior_MFMB6_mf_mb_prior_log_like], dim=-1)

MFMB6_mf_mb_prior_winning_model = MFMB6_mf_mb_prior_data_all_total_log_likes.argmin(dim=-1)

print("BCC2 planning wins", (MFMB6_mf_mb_prior_winning_model==0).sum())
print("BCC4 planning repetition wins", (MFMB6_mf_mb_prior_winning_model==1).sum())
print("BCC4 planning cached wins", (MFMB6_mf_mb_prior_winning_model==2).sum())
print("BCC6 planning repetition cached repetition wins", (MFMB6_mf_mb_prior_winning_model==3).sum())
print("MFMB4 MF MB wins", (MFMB6_mf_mb_prior_winning_model==4).sum())
print("MFMB6 MF MP Prior wins", (MFMB6_mf_mb_prior_winning_model==5).sum())

p_model = torch.nn.functional.softmax(-MFMB6_mf_mb_prior_data_all_total_log_likes, dim=-1)
print(p_model.shape)

print(p_model.mean(dim=0))

p_model_total = torch.nn.functional.softmax(-MFMB6_mf_mb_prior_data_all_total_log_likes.sum(dim=0), dim=-1)

print(p_model_total)

iu.calculate_exceedance_prob(-MFMB6_mf_mb_prior_data_all_total_log_likes)

BCC2 planning wins tensor(0)
BCC4 planning repetition wins tensor(0)
BCC4 planning cached wins tensor(0)
BCC6 planning repetition cached repetition wins tensor(1)
MFMB4 MF MB wins tensor(0)
MFMB6 MF MP Prior wins tensor(187)
torch.Size([188, 6])
tensor([2.7528e-12, 3.0233e-03, 2.8631e-11, 4.6490e-03, 2.0507e-03, 9.9028e-01])
tensor([0., 0., 0., 0., 0., 1.])
p model mean according to measure tensor([2.7528e-12, 3.0233e-03, 2.8631e-11, 4.6490e-03, 2.0507e-03, 9.9028e-01])
best model: tensor(5) exceedance prob tensor(1.)
is significantly different from uniform? TtestResult(statistic=np.float64(2498.154680457703), pvalue=np.float64(0.0), df=np.int64(499))
